
# LFH-aware multi-prefix stability check — Gemma2-9B-IT / AGNews

This notebook is the **coarse preliminary rebuttal version**, following the attached Gemma2-2B-IT candidate method rather than the later rank-stability/refined variants.

Experiment design:

1. Use Gemma2-9B-IT and the corresponding middle-layer residual SAE.
2. Sample AGNews test queries, label-balanced.
3. For each query, keep demonstrations 1–7 fixed and vary only demonstration 8 across eight balanced alternatives.
4. Measure query-side SAE pre/post-activations at token positions controlled by one variable only:
   ```python
   query_analysis_positions = [-2]
   ```
   Here `-1` is the final query-span token, `-2` is the second-final token, etc.
5. Select **label-specific** candidate features from the ordinary single-prefix baseline. For label `L`, the score is:
   ```text
   mean preactivation on gold-label-L query items
   - mean preactivation on all other query items
   ```
6. For each label, report the top-20 label-specific features from the baseline prefix and check whether they remain stable across final-demo variants.
7. Run the coarse causal validation: compare multi-prefix-stable versus LFH-sensitive label-feature pairs using active-feature ablation and feature-direction amplification.
8. Plot diagnostics with Plotly only.

The notebook does not read or write analysis caches. Re-running the notebook recomputes the activations from scratch.


In [1]:

# ============================================================
# 0. Imports and configuration
# ============================================================

import os
import re
import json
import math
import random
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import plotly.express as px
import plotly.graph_objects as go

from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import hf_hub_download
from IPython.display import display


def set_seed(seed: int):
    print("seed:", seed)
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.enabled = False
    torch.backends.cudnn.benchmark = False
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    os.environ["PYTHONHASHSEED"] = str(seed)


# -----------------------------
# User-facing experiment knobs
# -----------------------------
seed = 42
set_seed(seed)

MODEL_FAMILY = "gemma"
MODEL_NAME = "google/gemma-2-9b-it"
MODEL_SHORT_NAME = "Gemma2-9B-IT"
TARGET_LAYER = 20

# Gemma Scope 9B IT residual SAE, middle checkpoint.
SAE_BACKEND = "gemma_scope_npz"
SAE_REPO_ID = "google/gemma-scope-9b-it-res"
SAE_WIDTH = "width_16k"
SAE_AVERAGE_L0 = "average_l0_91"
SAE_FILENAME = f"layer_{TARGET_LAYER}/{SAE_WIDTH}/{SAE_AVERAGE_L0}/params.npz"

# Gemma Scope variants do not need Qwen high-residual-norm filtering.
FILTER_HIGH_NORM_TOKENS_FOR_FEATURE_EXTRACTION = False
HIGH_NORM_FILTER_MULTIPLIER = 10.0

# Multi-prefix stability experiment.
N_QUERY_TOTAL = 100              # label-balanced test queries; use smaller values for debugging
N_SHOT = 8                       # demonstrations per prefix
N_PREFIX_VARIANTS = 8            # with AGNews labels, 8 gives 2 final demos per label
FINAL_DEMO_SLOT = N_SHOT         # vary the last demonstration: Example 8

# Coarse attached-file method: top-20 baseline features and top-20 retention.
LABEL_SPECIFIC_TOP_N = 20        # top-20 features per label in the ordinary single-prefix baseline
RETENTION_TOP_K = 20             # retention is computed against the top-K label-specific ranking in each variant
STABLE_MIN_RETENTION = 0.9
STABLE_MIN_BASELINE_RATIO = 0.50

# One feature-selection token-position variable only.
# -1 is the final query-span token, -2 is the second-final token, etc.
query_analysis_positions = [-2]

# Optional local-control condition: vary an earlier demo while keeping the final demo fixed.
# This approximately doubles the number of forward passes, so it is disabled by default.
RUN_EARLIER_DEMO_CONTROL = False
EARLIER_DEMO_CONTROL_SLOT = 3

# Runtime. Gemma2-9B is larger than the attached 2B notebook, so the batch size is conservative.
DTYPE = torch.bfloat16
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 2

# Optional HF token for higher rate limits / gated checkpoints.
HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_HUB_TOKEN")
HF_KWARGS = {"token": HF_TOKEN} if HF_TOKEN else {}

print("MODEL_NAME:", MODEL_NAME)
print("MODEL_SHORT_NAME:", MODEL_SHORT_NAME)
print("MODEL_FAMILY:", MODEL_FAMILY)
print("TARGET_LAYER:", TARGET_LAYER)
print("SAE_BACKEND:", SAE_BACKEND)
print("SAE_REPO_ID:", SAE_REPO_ID)
print("SAE_FILENAME:", SAE_FILENAME)
print("FILTER_HIGH_NORM_TOKENS_FOR_FEATURE_EXTRACTION:", FILTER_HIGH_NORM_TOKENS_FOR_FEATURE_EXTRACTION)
print("LABEL_SPECIFIC_TOP_N:", LABEL_SPECIFIC_TOP_N)
print("RETENTION_TOP_K:", RETENTION_TOP_K)
print("query_analysis_positions:", query_analysis_positions)
print("BATCH_SIZE:", BATCH_SIZE)
print("DEVICE:", DEVICE)


/home/ikhyuncho23/ml-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


seed: 42
MODEL_NAME: google/gemma-2-9b-it
MODEL_SHORT_NAME: Gemma2-9B-IT
MODEL_FAMILY: gemma
TARGET_LAYER: 20
SAE_BACKEND: gemma_scope_npz
SAE_REPO_ID: google/gemma-scope-9b-it-res
SAE_FILENAME: layer_20/width_16k/average_l0_91/params.npz
FILTER_HIGH_NORM_TOKENS_FOR_FEATURE_EXTRACTION: False
LABEL_SPECIFIC_TOP_N: 20
RETENTION_TOP_K: 20
query_analysis_positions: [-2]
BATCH_SIZE: 2
DEVICE: cuda


In [2]:
# ============================================================
# 1. Load AGNews, tokenizer, model, and SAE
# ============================================================

label_to_words_agnews = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Technology",
}
word_to_label_agnews = {v: k for k, v in label_to_words_agnews.items()}
NUM_LABELS = len(label_to_words_agnews)


def load_agnews_splits():
    """Use the namespaced HF dataset id to avoid recent hf:// URI errors."""
    candidates = ["fancyzhx/ag_news", "ag_news"]
    errors = []
    for dataset_id in candidates:
        try:
            print(f"Loading AGNews from {dataset_id!r}...")
            return load_dataset(dataset_id, split=["train", "test"], **HF_KWARGS)
        except Exception as e:
            errors.append((dataset_id, type(e).__name__, str(e)))
            print(f"Failed to load {dataset_id!r}: {type(e).__name__}: {e}")
    raise RuntimeError("Could not load AGNews. Errors:\n" + "\n\n".join(map(str, errors)))


dataset = load_agnews_splits()
train_data_agnews = dataset[0]
test_data_agnews = dataset[1]

train_indices_by_label = defaultdict(list)
test_indices_by_label = defaultdict(list)
for i, row in enumerate(train_data_agnews):
    train_indices_by_label[int(row["label"])].append(i)
for i, row in enumerate(test_data_agnews):
    test_indices_by_label[int(row["label"])].append(i)

print("Train counts:", {label_to_words_agnews[k]: len(v) for k, v in train_indices_by_label.items()})
print("Test counts:", {label_to_words_agnews[k]: len(v) for k, v in test_indices_by_label.items()})


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    use_fast=True,
    **HF_KWARGS,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# Left padding makes causal labels natural and matches the previous notebooks.
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True,
    **HF_KWARGS,
)
model.eval()


def get_model_input_device(model):
    return model.get_input_embeddings().weight.device


def get_layer_device(model, layer_idx: int):
    block = model.get_submodule(f"model.layers.{layer_idx}")
    try:
        return next(block.parameters()).device
    except StopIteration:
        return get_model_input_device(model)


MODEL_INPUT_DEVICE = get_model_input_device(model)
TARGET_LAYER_DEVICE = get_layer_device(model, TARGET_LAYER)
print("MODEL_INPUT_DEVICE:", MODEL_INPUT_DEVICE)
print("TARGET_LAYER_DEVICE:", TARGET_LAYER_DEVICE)


class GemmaScopeJumpReLUSAE(torch.nn.Module):
    """Minimal inference wrapper for Gemma Scope JumpReLU residual-stream SAEs."""

    def __init__(self, params, device=DEVICE, dtype=DTYPE):
        super().__init__()
        self.device = torch.device(device)
        self.dtype = dtype

        def get_param(name, *aliases):
            for key in (name,) + aliases:
                if key in params:
                    value = params[key]
                    if isinstance(value, np.ndarray):
                        value = torch.from_numpy(value)
                    return value.detach() if torch.is_tensor(value) else torch.tensor(value)
            raise KeyError(f"Missing SAE parameter {name!r}. Available keys: {sorted(params.keys())}")

        W_enc = get_param("W_enc")
        W_dec = get_param("W_dec")
        b_enc = get_param("b_enc")
        b_dec = get_param("b_dec")
        threshold = get_param("threshold", "thresholds")

        if W_enc.ndim != 2 or W_dec.ndim != 2:
            raise ValueError(f"Expected 2D W_enc/W_dec, got {tuple(W_enc.shape)} and {tuple(W_dec.shape)}")

        d_model = int(W_enc.shape[0])
        n_features = int(W_enc.shape[1])

        if W_dec.shape == (d_model, n_features):
            W_dec = W_dec.T.contiguous()
        if W_dec.shape != (n_features, d_model):
            raise ValueError(f"Expected W_dec shape {(n_features, d_model)}, got {tuple(W_dec.shape)}")
        if b_enc.numel() != n_features:
            raise ValueError(f"Expected b_enc length {n_features}, got {b_enc.numel()}")
        if b_dec.numel() != d_model:
            raise ValueError(f"Expected b_dec length {d_model}, got {b_dec.numel()}")
        if threshold.numel() != n_features:
            raise ValueError(f"Expected threshold length {n_features}, got {threshold.numel()}")

        self.register_buffer("W_enc", W_enc.contiguous().to(device=self.device, dtype=self.dtype))
        self.register_buffer("W_dec", W_dec.contiguous().to(device=self.device, dtype=self.dtype))
        self.register_buffer("b_enc", b_enc.reshape(-1).contiguous().to(device=self.device, dtype=self.dtype))
        self.register_buffer("b_dec", b_dec.reshape(-1).contiguous().to(device=self.device, dtype=self.dtype))
        self.register_buffer("threshold", threshold.reshape(-1).contiguous().to(device=self.device, dtype=self.dtype))

        self.k = None
        self.d_model = d_model
        self.n_features = n_features

    def process_sae_in(self, x: torch.Tensor) -> torch.Tensor:
        # Gemma Scope folds any preprocessing into the encoder parameters for these checkpoints.
        return x

    def encode_pre(self, x: torch.Tensor) -> torch.Tensor:
        x = x.to(device=self.device, dtype=self.dtype)
        return self.process_sae_in(x) @ self.W_enc + self.b_enc

    def activation_fn(self, pre: torch.Tensor) -> torch.Tensor:
        return torch.where(pre > self.threshold, torch.relu(pre), torch.zeros_like(pre))

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.activation_fn(self.encode_pre(x))

    def decode(self, acts: torch.Tensor) -> torch.Tensor:
        acts = acts.to(device=self.device, dtype=self.dtype)
        return acts @ self.W_dec + self.b_dec


class QwenBatchTopKSAE(torch.nn.Module):
    """
    Minimal inference wrapper for Adam Karvonen's Qwen3 BatchTopK SAEs.

    The checkpoint has dictionary_learning-style keys:
      b_dec, k, threshold, decoder.weight, encoder.weight, encoder.bias

    This wrapper exposes the same interface used by the Gemma Scope wrapper:
      W_enc, W_dec, b_enc, b_dec, threshold, encode_pre, activation_fn, encode, decode.
    """

    def __init__(self, state_dict, device=DEVICE, dtype=DTYPE):
        super().__init__()
        self.device = torch.device(device)
        self.dtype = dtype

        if isinstance(state_dict, dict) and "state_dict" in state_dict:
            state_dict = state_dict["state_dict"]
        if isinstance(state_dict, dict) and "model" in state_dict and isinstance(state_dict["model"], dict):
            state_dict = state_dict["model"]

        b_dec = state_dict["b_dec"].detach()
        b_enc = state_dict.get("encoder.bias", state_dict.get("b_enc")).detach()
        enc_w = state_dict.get("encoder.weight", state_dict.get("W_enc")).detach()
        dec_w = state_dict.get("decoder.weight", state_dict.get("W_dec")).detach()
        threshold = state_dict["threshold"]
        threshold = threshold.detach() if torch.is_tensor(threshold) else torch.tensor(threshold)
        k = state_dict.get("k", None)
        if torch.is_tensor(k):
            k = int(k.item())

        d_model = int(b_dec.numel())
        n_features = int(b_enc.numel())

        if enc_w.shape == (n_features, d_model):
            W_enc = enc_w.T.contiguous()
        elif enc_w.shape == (d_model, n_features):
            W_enc = enc_w.contiguous()
        else:
            raise ValueError(
                f"Unexpected encoder.weight shape {tuple(enc_w.shape)} "
                f"for d_model={d_model}, n_features={n_features}"
            )

        if dec_w.shape == (d_model, n_features):
            W_dec = dec_w.T.contiguous()
        elif dec_w.shape == (n_features, d_model):
            W_dec = dec_w.contiguous()
        else:
            raise ValueError(
                f"Unexpected decoder.weight shape {tuple(dec_w.shape)} "
                f"for d_model={d_model}, n_features={n_features}"
            )

        self.register_buffer("W_enc", W_enc.to(device=self.device, dtype=self.dtype))
        self.register_buffer("W_dec", W_dec.to(device=self.device, dtype=self.dtype))
        self.register_buffer("b_enc", b_enc.to(device=self.device, dtype=self.dtype))
        self.register_buffer("b_dec", b_dec.to(device=self.device, dtype=self.dtype))
        self.register_buffer("threshold", threshold.to(device=self.device, dtype=self.dtype))

        self.k = k
        self.d_model = d_model
        self.n_features = n_features

    def process_sae_in(self, x: torch.Tensor) -> torch.Tensor:
        return x - self.b_dec

    def encode_pre(self, x: torch.Tensor) -> torch.Tensor:
        x = x.to(device=self.device, dtype=self.dtype)
        return self.process_sae_in(x) @ self.W_enc + self.b_enc

    def activation_fn(self, pre: torch.Tensor) -> torch.Tensor:
        return torch.where(pre > self.threshold, pre, torch.zeros_like(pre))

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.activation_fn(self.encode_pre(x))

    def decode(self, acts: torch.Tensor) -> torch.Tensor:
        acts = acts.to(device=self.device, dtype=self.dtype)
        return acts @ self.W_dec + self.b_dec


def load_gemma_scope_jumprelu_sae(repo_id: str, filename: str, device=DEVICE, dtype=DTYPE):
    print("Downloading/loading Gemma Scope SAE checkpoint:", repo_id, filename)
    path = hf_hub_download(repo_id=repo_id, filename=filename, **HF_KWARGS)
    with np.load(path, allow_pickle=False) as params_np:
        params = {k: torch.from_numpy(params_np[k]) for k in params_np.files}
    sae = GemmaScopeJumpReLUSAE(params, device=device, dtype=dtype)
    print(
        f"Loaded SAE: d_model={sae.d_model}, n_features={sae.n_features}, "
        f"mean_threshold={float(sae.threshold.detach().float().mean().cpu()):.4f}"
    )
    return sae


def load_qwen_batchtopk_sae(repo_id: str, filename: str, device=DEVICE, dtype=DTYPE):
    print("Downloading/loading Qwen BatchTopK SAE checkpoint:", repo_id, filename)
    path = hf_hub_download(repo_id=repo_id, filename=filename, **HF_KWARGS)
    try:
        state = torch.load(path, map_location="cpu", weights_only=True)
    except TypeError:
        state = torch.load(path, map_location="cpu")
    sae = QwenBatchTopKSAE(state, device=device, dtype=dtype)
    print(
        f"Loaded SAE: d_model={sae.d_model}, n_features={sae.n_features}, "
        f"k={sae.k}, mean_threshold={float(sae.threshold.detach().float().mean().cpu()):.4f}"
    )
    return sae


def load_configured_sae():
    if SAE_BACKEND == "gemma_scope_npz":
        return load_gemma_scope_jumprelu_sae(
            repo_id=SAE_REPO_ID,
            filename=SAE_FILENAME,
            device=TARGET_LAYER_DEVICE,
            dtype=DTYPE,
        )
    if SAE_BACKEND == "qwen_batchtopk":
        return load_qwen_batchtopk_sae(
            repo_id=SAE_REPO_ID,
            filename=SAE_FILENAME,
            device=TARGET_LAYER_DEVICE,
            dtype=DTYPE,
        )
    raise ValueError(f"Unknown SAE_BACKEND={SAE_BACKEND!r}")


sae = load_configured_sae()
print("SAE device:", sae.device)


Loading AGNews from 'fancyzhx/ag_news'...


Train counts: {'Business': 30000, 'Technology': 30000, 'Sports': 30000, 'World': 30000}
Test counts: {'Business': 1900, 'Technology': 1900, 'Sports': 1900, 'World': 1900}


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|████████████████████████████████| 464/464 [00:00<00:00, 486.39it/s]


MODEL_INPUT_DEVICE: cuda:0
TARGET_LAYER_DEVICE: cuda:0
Downloading/loading Gemma Scope SAE checkpoint: google/gemma-scope-9b-it-res layer_20/width_16k/average_l0_91/params.npz
Loaded SAE: d_model=3584, n_features=16384, mean_threshold=5.3436
SAE device: cuda:0


In [3]:

# ============================================================
# 2. Prompt sampling and rendering
# ============================================================

rng = random.Random(seed)

INSTRUCTION = """You are given news articles and their topics.
Valid topics: World, Sports, Business, Technology.
For each example, identify the correct topic."""


def balanced_label_sequence(n: int, num_labels: int = NUM_LABELS, rng: random.Random | None = None):
    """Return labels as evenly as possible, shuffled."""
    rng = rng or random
    base = n // num_labels
    rem = n % num_labels
    labels = []
    for lab in range(num_labels):
        labels.extend([lab] * base)
    extra = list(range(num_labels))
    rng.shuffle(extra)
    labels.extend(extra[:rem])
    rng.shuffle(labels)
    return labels


def sample_train_indices_for_labels(label_seq, rng: random.Random, exclude=None):
    exclude = set(exclude or [])
    chosen = []
    used = set(exclude)
    for lab in label_seq:
        pool = [idx for idx in train_indices_by_label[int(lab)] if idx not in used]
        if not pool:
            raise RuntimeError(f"No available training examples for label {lab} after exclusions.")
        idx = rng.choice(pool)
        chosen.append(idx)
        used.add(idx)
    return chosen


def sample_label_balanced_test_indices(n_total: int, rng: random.Random):
    label_seq = balanced_label_sequence(n_total, num_labels=NUM_LABELS, rng=rng)
    out = []
    used = set()
    for lab in label_seq:
        pool = [idx for idx in test_indices_by_label[int(lab)] if idx not in used]
        idx = rng.choice(pool)
        out.append(idx)
        used.add(idx)
    return out


def format_demo(demo_number: int, row) -> str:
    return (
        f"Example {demo_number}:\n"
        f"News article:\n{row['text']}\n"
        f"Topic:\n{label_to_words_agnews[int(row['label'])]}"
    )


def render_single_user_prompt(demo_rows, query_row):
    """
    Render all demonstrations and the test query inside one model-native user prompt.

    Gemma uses one <start_of_turn>user block plus the model-generation header.
    Qwen uses one system message plus one user message plus the assistant-generation header.
    In both cases, all demonstrations are in a single user prompt; individual demos are not wrapped.

    Returns:
      prompt: full model input string
      query_char_span: [start, end) character span of the query block inside prompt
    """
    if len(demo_rows) != N_SHOT:
        raise ValueError(f"Expected {N_SHOT} demos, got {len(demo_rows)}")

    demo_text = "\n\n".join(format_demo(i + 1, row) for i, row in enumerate(demo_rows))
    query_block = f"Test article:\n{query_row['text']}\nTopic:\n"

    if MODEL_FAMILY == "gemma":
        user_content = f"{INSTRUCTION}\n\n{demo_text}\n\n{query_block}"
        prompt = f"<bos><start_of_turn>user\n{user_content}<end_of_turn>\n<start_of_turn>model\n"

        # Sanity checks: one user turn, one model-generation header, all demos in the user turn.
        if prompt.count("<start_of_turn>user") != 1:
            raise AssertionError("Prompt should contain exactly one user turn.")
        if prompt.count("<start_of_turn>model") != 1:
            raise AssertionError("Prompt should contain exactly one model-generation header.")
        if prompt.count("<end_of_turn>") != 1:
            raise AssertionError("Prompt should contain exactly one user-turn terminator.")

    elif MODEL_FAMILY == "qwen":
        user_content = f"{demo_text}\n\n{query_block}"
        messages = [
            {"role": "system", "content": INSTRUCTION},
            {"role": "user", "content": user_content},
        ]
        try:
            prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
        except TypeError:
            # Older transformers/tokenizers may not expose enable_thinking.
            # In that case, include Qwen's non-thinking directive in the system content.
            messages = [
                {"role": "system", "content": INSTRUCTION + "\n/no_think"},
                {"role": "user", "content": user_content},
            ]
            prompt = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )

        if prompt.count(query_block) != 1:
            raise AssertionError("Could not uniquely locate the query block in the Qwen chat-template prompt.")
    else:
        raise ValueError(f"Unknown MODEL_FAMILY={MODEL_FAMILY!r}")

    query_start = prompt.index(query_block)
    query_end = query_start + len(query_block)

    if len(re.findall(r"(?m)^Example\s+\d+:", prompt)) != N_SHOT:
        raise AssertionError(f"Prompt should contain exactly {N_SHOT} numbered demo headers.")

    return prompt, (query_start, query_end)


def build_final_demo_variant_prompts():
    """
    Build Q x V prompts using the coarse attached-file method.

    For each fixed query:
      - demos 1..7 are fixed for that query;
      - demo 8 varies across N_PREFIX_VARIANTS alternatives;
      - final-demo labels are as balanced as possible.
    """
    test_indices = sample_label_balanced_test_indices(N_QUERY_TOTAL, rng)
    prompts = []
    metadata = []

    for q_local_idx, test_idx in enumerate(test_indices):
        query_row = test_data_agnews[int(test_idx)]

        fixed_label_seq = balanced_label_sequence(N_SHOT - 1, rng=rng)
        fixed_demo_indices = sample_train_indices_for_labels(fixed_label_seq, rng=rng)

        final_label_seq = balanced_label_sequence(N_PREFIX_VARIANTS, rng=rng)
        final_demo_indices = sample_train_indices_for_labels(
            final_label_seq,
            rng=rng,
            exclude=fixed_demo_indices,
        )

        fixed_demo_rows = [train_data_agnews[int(i)] for i in fixed_demo_indices]

        for variant_idx, final_demo_idx in enumerate(final_demo_indices):
            demo_indices = list(fixed_demo_indices) + [int(final_demo_idx)]
            demo_rows = fixed_demo_rows + [train_data_agnews[int(final_demo_idx)]]
            prompt, query_char_span = render_single_user_prompt(demo_rows, query_row)
            prompts.append(prompt)
            metadata.append({
                "query_local_idx": int(q_local_idx),
                "variant_idx": int(variant_idx),
                "query_test_idx": int(test_idx),
                "query_label": int(query_row["label"]),
                "query_label_word": label_to_words_agnews[int(query_row["label"])],
                "fixed_demo_indices": [int(x) for x in fixed_demo_indices],
                "fixed_demo_labels": [int(train_data_agnews[int(x)]["label"]) for x in fixed_demo_indices],
                "final_demo_idx": int(final_demo_idx),
                "final_demo_label": int(train_data_agnews[int(final_demo_idx)]["label"]),
                "final_demo_label_word": label_to_words_agnews[int(train_data_agnews[int(final_demo_idx)]["label"])],
                "query_char_start": int(query_char_span[0]),
                "query_char_end": int(query_char_span[1]),
            })

    return prompts, metadata


prompts, prompt_metadata = build_final_demo_variant_prompts()
print(f"Built {len(prompts)} prompts = {N_QUERY_TOTAL} queries x {N_PREFIX_VARIANTS} final-demo variants.")
print("First prompt excerpt:\n")
print(prompts[0][:2000])

meta_df = pd.DataFrame(prompt_metadata)
print("\nQuery label counts:")
display(meta_df.drop_duplicates("query_local_idx")["query_label_word"].value_counts().sort_index().to_frame("count"))
print("Final-demo label counts across all variants:")
display(meta_df["final_demo_label_word"].value_counts().sort_index().to_frame("count"))


Built 800 prompts = 100 queries x 8 final-demo variants.
First prompt excerpt:

<bos><start_of_turn>user
You are given news articles and their topics.
Valid topics: World, Sports, Business, Technology.
For each example, identify the correct topic.

Example 1:
News article:
Hamm #39;s legacy should be for Olympic ideal Even though he has fled the perceived hostile atmosphere of Athens and the Olympic village for more friendly terrain in the United States, there is still time for Paul Hamm to do what #39;s right.
Topic:
Sports

Example 2:
News article:
Greenspan rejects talk of throwing dollar a lifeline ALAN GREENSPAN, Chairman of the US Federal Reserve, undercut lingering hopes of staging an operation to support the dollar at a meeting of bankers in Frankfurt yesterday 
Topic:
Business

Example 3:
News article:
Google Desktop Search app lets you 'google' your Windows PC Google Inc. unveiled a desktop search application that lets users catalog and search through e-mail, files, Web histo

,count
query_label_word,
Business,25
Sports,25
Technology,25
World,25


Final-demo label counts across all variants:


,count
final_demo_label_word,
Business,200
Sports,200
Technology,200
World,200


In [4]:

# ============================================================
# 3. Token-span helpers and model/SAE extraction
# ============================================================


def token_positions_for_char_span(prompt: str, char_start: int, char_end: int):
    """Map a prompt character span to unpadded token indices."""
    enc = tokenizer(
        prompt,
        add_special_tokens=False,
        return_offsets_mapping=True,
    )
    offsets = enc["offset_mapping"]
    positions = []
    for tok_idx, (s, e) in enumerate(offsets):
        # Skip empty special offsets if any appear.
        if e <= s:
            continue
        # Token overlaps the query span.
        if max(int(s), int(char_start)) < min(int(e), int(char_end)):
            positions.append(tok_idx)
    if not positions:
        raise RuntimeError("No token positions found for query character span.")
    return positions


def select_query_positions(all_query_positions):
    """
    Select query-span token positions using query_analysis_positions only.

    Offset semantics:
      -1 -> final query-span token
      -2 -> second-final query-span token
      -3 -> third-final query-span token
    """
    selected = []
    n = len(all_query_positions)
    for off in query_analysis_positions:
        off = int(off)
        if off >= 0:
            raise ValueError("query_analysis_positions must contain negative offsets, e.g. [-1, -2].")
        idx = n + off
        if idx < 0 or idx >= n:
            raise IndexError(f"Offset {off} is out of range for query span with {n} tokens.")
        selected.append(all_query_positions[idx])
    return selected


# Precompute selected unpadded token positions for every prompt.
for meta, prompt in zip(prompt_metadata, prompts):
    all_positions = token_positions_for_char_span(
        prompt=prompt,
        char_start=meta["query_char_start"],
        char_end=meta["query_char_end"],
    )
    meta["all_query_token_positions"] = [int(x) for x in all_positions]
    meta["selected_query_token_positions"] = [int(x) for x in select_query_positions(all_positions)]
    meta["num_query_tokens"] = int(len(all_positions))
    meta["num_selected_query_tokens"] = int(len(meta["selected_query_token_positions"]))

print("Selected query-token counts:")
display(pd.Series([m["num_selected_query_tokens"] for m in prompt_metadata]).describe().to_frame("selected_query_tokens"))
print("Example selected offsets:", query_analysis_positions)
print("First prompt selected unpadded token positions:", prompt_metadata[0]["selected_query_token_positions"])


def clear_all_hooks(model: torch.nn.Module):
    for module in model.modules():
        module._forward_hooks.clear()
        module._forward_pre_hooks.clear()
        module._backward_hooks.clear()


@torch.no_grad()
def get_layer_output(model, input_ids, attention_mask, layer_idx: int):
    """Run a forward pass and capture model.layers[layer_idx] output, shape [B, S, D]."""
    captured = {}

    def hook_fn(module, inputs, output):
        captured["x"] = output[0].detach() if isinstance(output, tuple) else output.detach()
        return output

    block = model.get_submodule(f"model.layers.{layer_idx}")
    handle = block.register_forward_hook(hook_fn)
    try:
        _ = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)
    finally:
        handle.remove()

    if "x" not in captured:
        raise RuntimeError(f"Failed to capture output from layer {layer_idx}")
    return captured["x"]



@torch.no_grad()
def sae_encode_pre_for_positions(hidden_row: torch.Tensor, selected_positions):
    """
    Encode SAE preactivations for selected positions from one prompt's hidden states.

    For Qwen BatchTopK SAEs, optionally zero out high-residual-norm tokens using the
    same median-norm rule used in the Qwen SAE release. Gemma variants keep this off.
    """
    positions = [int(p) for p in selected_positions]
    x = hidden_row[positions, :]
    pre = sae.encode_pre(x).float()

    if FILTER_HIGH_NORM_TOKENS_FOR_FEATURE_EXTRACTION:
        norms = hidden_row.detach().float().norm(dim=-1)
        median_norm = norms.median().clamp_min(1e-6)
        high_norm_mask = norms[positions] > (HIGH_NORM_FILTER_MULTIPLIER * median_norm)
        if bool(high_norm_mask.any().item()):
            pre = pre.clone()
            pre[high_norm_mask.to(pre.device)] = 0.0

    return pre




def apply_optional_high_norm_filter_to_pre(x: torch.Tensor, pre: torch.Tensor) -> torch.Tensor:
    """
    Optionally zero SAE preactivations for abnormally high-residual-norm tokens.

    This is enabled for Qwen BatchTopK SAEs and disabled for Gemma Scope SAEs.
    """
    if not FILTER_HIGH_NORM_TOKENS_FOR_FEATURE_EXTRACTION:
        return pre
    if x.ndim < 2:
        return pre
    norms = x.detach().float().norm(dim=-1)
    median_norm = norms.median().clamp_min(1e-6)
    high_norm_mask = norms > (HIGH_NORM_FILTER_MULTIPLIER * median_norm)
    if high_norm_mask.any():
        pre = pre.masked_fill(high_norm_mask.to(pre.device)[..., None], 0.0)
    return pre

@torch.no_grad()
def collect_query_feature_stats_for_prompts(prompts, metadata):
    """
    Return per-prompt query-side feature statistics:
      pre_by_prompt:  [P, F] mean SAE pre-activation over selected query tokens
      post_by_prompt: [P, F] mean SAE post-activation over selected query tokens
      freq_by_prompt: [P, F] fraction of selected query tokens with post-activation > 0
    """

    n_prompts = len(prompts)
    n_features = int(sae.n_features)
    pre_by_prompt = np.zeros((n_prompts, n_features), dtype=np.float32)
    post_by_prompt = np.zeros((n_prompts, n_features), dtype=np.float32)
    freq_by_prompt = np.zeros((n_prompts, n_features), dtype=np.float32)

    model_device = get_model_input_device(model)
    model.eval()
    sae.eval()

    for start in range(0, n_prompts, BATCH_SIZE):
        end = min(start + BATCH_SIZE, n_prompts)
        batch_prompts = prompts[start:end]
        batch_meta = metadata[start:end]

        enc = tokenizer(
            batch_prompts,
            padding=True,
            return_tensors="pt",
            add_special_tokens=False,
        )
        input_ids = enc["input_ids"].to(model_device)
        attention_mask = enc["attention_mask"].to(model_device)

        hidden = get_layer_output(model, input_ids, attention_mask, TARGET_LAYER)  # [B, S, D]
        batch_seq_len = int(input_ids.shape[1])

        for b, meta in enumerate(batch_meta):
            unpadded_len = int(attention_mask[b].sum().item())
            pad_shift = batch_seq_len - unpadded_len
            selected_positions = [pad_shift + int(pos) for pos in meta["selected_query_token_positions"]]
            selected_positions = [pos for pos in selected_positions if 0 <= pos < batch_seq_len]
            if not selected_positions:
                raise RuntimeError(f"No valid selected token positions for prompt index {start + b}.")

            pre = sae_encode_pre_for_positions(hidden[b], selected_positions)
            post = sae.activation_fn(pre.to(dtype=sae.dtype)).float()

            pre_by_prompt[start + b] = pre.mean(dim=0).detach().cpu().numpy()
            post_by_prompt[start + b] = post.mean(dim=0).detach().cpu().numpy()
            freq_by_prompt[start + b] = (post > 0).float().mean(dim=0).detach().cpu().numpy()

        del hidden, input_ids, attention_mask, enc

        print(f"Processed prompts {end}/{n_prompts}")

    return pre_by_prompt, post_by_prompt, freq_by_prompt, metadata


Selected query-token counts:


,selected_query_tokens
count,800.0
mean,1.0
std,0.0
min,1.0
25%,1.0
50%,1.0
75%,1.0
max,1.0


Example selected offsets: [-2]
First prompt selected unpadded token positions: [577]


In [5]:

# ============================================================
# 4. Run final-demo-variant extraction
# ============================================================

pre_by_prompt, post_by_prompt, freq_by_prompt, prompt_metadata_loaded = collect_query_feature_stats_for_prompts(
    prompts=prompts,
    metadata=prompt_metadata,
)

# Reshape from [Q*V, F] to [Q, V, F]. Prompts were built query-major, then variant-major.
n_features = pre_by_prompt.shape[-1]
pre_by_query_variant = pre_by_prompt.reshape(N_QUERY_TOTAL, N_PREFIX_VARIANTS, n_features)
post_by_query_variant = post_by_prompt.reshape(N_QUERY_TOTAL, N_PREFIX_VARIANTS, n_features)
freq_by_query_variant = freq_by_prompt.reshape(N_QUERY_TOTAL, N_PREFIX_VARIANTS, n_features)

print("pre_by_query_variant:", pre_by_query_variant.shape)
print("post_by_query_variant:", post_by_query_variant.shape)
print("freq_by_query_variant:", freq_by_query_variant.shape)


Processed prompts 2/800
Processed prompts 4/800
Processed prompts 6/800
Processed prompts 8/800
Processed prompts 10/800
Processed prompts 12/800
Processed prompts 14/800
Processed prompts 16/800
Processed prompts 18/800
Processed prompts 20/800
Processed prompts 22/800
Processed prompts 24/800
Processed prompts 26/800
Processed prompts 28/800
Processed prompts 30/800
Processed prompts 32/800
Processed prompts 34/800
Processed prompts 36/800
Processed prompts 38/800
Processed prompts 40/800
Processed prompts 42/800
Processed prompts 44/800
Processed prompts 46/800
Processed prompts 48/800
Processed prompts 50/800
Processed prompts 52/800
Processed prompts 54/800
Processed prompts 56/800
Processed prompts 58/800
Processed prompts 60/800
Processed prompts 62/800
Processed prompts 64/800
Processed prompts 66/800
Processed prompts 68/800
Processed prompts 70/800
Processed prompts 72/800
Processed prompts 74/800
Processed prompts 76/800
Processed prompts 78/800
Processed prompts 80/800
Proc

Processed prompts 638/800
Processed prompts 640/800
Processed prompts 642/800
Processed prompts 644/800
Processed prompts 646/800
Processed prompts 648/800
Processed prompts 650/800
Processed prompts 652/800
Processed prompts 654/800
Processed prompts 656/800
Processed prompts 658/800
Processed prompts 660/800
Processed prompts 662/800
Processed prompts 664/800
Processed prompts 666/800
Processed prompts 668/800
Processed prompts 670/800
Processed prompts 672/800
Processed prompts 674/800
Processed prompts 676/800
Processed prompts 678/800
Processed prompts 680/800
Processed prompts 682/800
Processed prompts 684/800
Processed prompts 686/800
Processed prompts 688/800
Processed prompts 690/800
Processed prompts 692/800
Processed prompts 694/800
Processed prompts 696/800
Processed prompts 698/800
Processed prompts 700/800
Processed prompts 702/800
Processed prompts 704/800
Processed prompts 706/800
Processed prompts 708/800
Processed prompts 710/800
Processed prompts 712/800
Processed pr

In [6]:

# ============================================================
# 5. Label-specific feature selection and multi-prefix stability
# ============================================================

# Query labels in the same query-major order used by pre_by_query_variant/post_by_query_variant.
query_meta_df = meta_df.drop_duplicates("query_local_idx").sort_values("query_local_idx").reset_index(drop=True)
query_labels = query_meta_df["query_label"].to_numpy(dtype=np.int64)
if len(query_labels) != N_QUERY_TOTAL:
    raise AssertionError(f"Expected {N_QUERY_TOTAL} query labels, got {len(query_labels)}")

print("Query label counts used for label-specific scoring:")
display(pd.Series(query_labels).map(label_to_words_agnews).value_counts().sort_index().to_frame("count"))


def label_vs_other_means(arr_qvf: np.ndarray, labels: np.ndarray):
    """
    Compute label-vs-other means for arrays shaped [Q, V, F].

    Returns:
      label_mean: [L, V, F]
      other_mean: [L, V, F]
    """
    if arr_qvf.shape[0] != labels.shape[0]:
        raise ValueError(f"arr_qvf has {arr_qvf.shape[0]} queries but labels has {labels.shape[0]}")
    label_mean = np.zeros((NUM_LABELS, arr_qvf.shape[1], arr_qvf.shape[2]), dtype=np.float32)
    other_mean = np.zeros_like(label_mean)
    for lab in range(NUM_LABELS):
        mask = labels == lab
        other = ~mask
        if mask.sum() == 0 or other.sum() == 0:
            raise RuntimeError(f"Need at least one label-{lab} and one non-label-{lab} query.")
        label_mean[lab] = arr_qvf[mask].mean(axis=0)
        other_mean[lab] = arr_qvf[other].mean(axis=0)
    return label_mean, other_mean


# Label-specific score requested by the user:
#   average preactivation on label L - average preactivation on all other labels.
label_mean_pre, other_mean_pre = label_vs_other_means(pre_by_query_variant, query_labels)
label_mean_post, other_mean_post = label_vs_other_means(post_by_query_variant, query_labels)
label_mean_freq, other_mean_freq = label_vs_other_means(freq_by_query_variant, query_labels)
label_specific_pre_gap = label_mean_pre - other_mean_pre  # [label, variant, feature]

# Rank every feature by label-specific preactivation gap within each label and prefix variant.
label_variant_ranks = np.empty_like(label_specific_pre_gap, dtype=np.int32)  # [L, V, F]
for lab in range(NUM_LABELS):
    for v in range(N_PREFIX_VARIANTS):
        order = np.argsort(-label_specific_pre_gap[lab, v])
        ranks_v = np.empty(n_features, dtype=np.int32)
        ranks_v[order] = np.arange(1, n_features + 1, dtype=np.int32)
        label_variant_ranks[lab, v] = ranks_v


def summarize_label_specific_feature(lab: int, feat: int, rank: int):
    scores_v = label_specific_pre_gap[lab, :, feat]
    ranks_v = label_variant_ranks[lab, :, feat]
    baseline = float(scores_v[0])
    mean_score = float(scores_v.mean())
    std_score = float(scores_v.std(ddof=0))
    min_score = float(scores_v.min())
    max_score = float(scores_v.max())
    score_range = float(max_score - min_score)
    eps = 1e-12

    if baseline > eps:
        min_over_baseline = float(min_score / baseline)
        mean_over_baseline = float(mean_score / baseline)
    else:
        min_over_baseline = np.nan
        mean_over_baseline = np.nan

    retention_rate = float((ranks_v <= RETENTION_TOP_K).mean())
    stable = bool(
        (retention_rate >= STABLE_MIN_RETENTION)
        and np.isfinite(min_over_baseline)
        and (min_over_baseline >= STABLE_MIN_BASELINE_RATIO)
    )

    return {
        "label": int(lab),
        "label_word": label_to_words_agnews[int(lab)],
        "label_specific_rank": int(rank),
        "feature_idx": int(feat),
        "baseline_label_specific_pre_gap": baseline,
        "mean_label_specific_pre_gap_across_prefixes": mean_score,
        "std_label_specific_pre_gap_across_prefixes": std_score,
        "min_label_specific_pre_gap_across_prefixes": min_score,
        "max_label_specific_pre_gap_across_prefixes": max_score,
        "range_label_specific_pre_gap_across_prefixes": score_range,
        "mean_over_baseline": mean_over_baseline,
        "min_over_baseline": min_over_baseline,
        f"retention_rate_top_{RETENTION_TOP_K}": retention_rate,
        "worst_variant_rank": int(ranks_v.max()),
        "best_variant_rank": int(ranks_v.min()),
        "stable_by_rule": stable,
        "baseline_mean_pre_label": float(label_mean_pre[lab, 0, feat]),
        "baseline_mean_pre_other": float(other_mean_pre[lab, 0, feat]),
        "baseline_mean_post_label": float(label_mean_post[lab, 0, feat]),
        "baseline_mean_post_other": float(other_mean_post[lab, 0, feat]),
        "baseline_activation_frequency_label": float(label_mean_freq[lab, 0, feat]),
        "baseline_activation_frequency_other": float(other_mean_freq[lab, 0, feat]),
        "mean_pre_label_across_prefixes": float(label_mean_pre[lab, :, feat].mean()),
        "mean_pre_other_across_prefixes": float(other_mean_pre[lab, :, feat].mean()),
        "mean_post_label_across_prefixes": float(label_mean_post[lab, :, feat].mean()),
        "mean_post_other_across_prefixes": float(other_mean_post[lab, :, feat].mean()),
        "activation_frequency_label_across_prefixes": float(label_mean_freq[lab, :, feat].mean()),
        "activation_frequency_other_across_prefixes": float(other_mean_freq[lab, :, feat].mean()),
    }


label_top_features = {}
rows = []
for lab in range(NUM_LABELS):
    baseline_scores = label_specific_pre_gap[lab, 0]  # ordinary single-prefix setting, variant 0
    order = np.argsort(-baseline_scores)
    top_feats = [int(x) for x in order[:LABEL_SPECIFIC_TOP_N]]
    label_top_features[lab] = top_feats
    for rank, feat in enumerate(top_feats, start=1):
        rows.append(summarize_label_specific_feature(lab=lab, feat=feat, rank=rank))

label_specific_stability_df = pd.DataFrame(rows)
label_specific_stability_df = label_specific_stability_df.sort_values(
    by=["label", "label_specific_rank"],
    ascending=[True, True],
).reset_index(drop=True)

lfh_sensitive_label_specific_df = label_specific_stability_df[~label_specific_stability_df["stable_by_rule"]].sort_values(
    by=["label", "label_specific_rank"],
    ascending=[True, True],
).reset_index(drop=True)

print(f"Top-{LABEL_SPECIFIC_TOP_N} label-specific features per label, selected by baseline preactivation gap.")
display(label_specific_stability_df)

print("\nOriginally top-ranked label-specific features flagged as LFH-sensitive by the simple multi-prefix rule:")
display(lfh_sensitive_label_specific_df)

print("\nPer-label tables:")
for lab in range(NUM_LABELS):
    label_word = label_to_words_agnews[lab]
    print("=" * 100)
    print(f"{label_word}: top-{LABEL_SPECIFIC_TOP_N} label-specific features")
    display(label_specific_stability_df[label_specific_stability_df["label"] == lab].reset_index(drop=True))

# Union of all top label-specific features, useful for optional controls.
candidate_label_feature_pairs = [
    (int(row["label"]), int(row["feature_idx"]))
    for _, row in label_specific_stability_df.iterrows()
]
candidate_feature_ids = sorted({feat for _, feat in candidate_label_feature_pairs})
print(f"Union of selected features across labels: {len(candidate_feature_ids)} unique features.")


Query label counts used for label-specific scoring:


,count
Business,25
Sports,25
Technology,25
World,25


Top-20 label-specific features per label, selected by baseline preactivation gap.


,label,label_word,label_specific_rank,feature_idx,baseline_label_specific_pre_gap,mean_label_specific_pre_gap_across_prefixes,std_label_specific_pre_gap_across_prefixes,min_label_specific_pre_gap_across_prefixes,max_label_specific_pre_gap_across_prefixes,range_label_specific_pre_gap_across_prefixes,...,baseline_mean_post_label,baseline_mean_post_other,baseline_activation_frequency_label,baseline_activation_frequency_other,mean_pre_label_across_prefixes,mean_pre_other_across_prefixes,mean_post_label_across_prefixes,mean_post_other_across_prefixes,activation_frequency_label_across_prefixes,activation_frequency_other_across_prefixes
0,0,World,1,13844,22.572865,21.723679,1.048429,19.794817,23.019375,3.224558,...,8.417500,0.925417,0.60,0.080000,5.061728,-16.661951,7.690000,0.863125,0.535,0.071667
1,0,World,2,3994,17.208542,18.228321,0.666845,17.208542,19.366875,2.158333,...,22.834999,8.120417,1.00,0.520000,23.457188,5.228868,23.387655,8.265417,0.985,0.526667
2,0,World,3,14744,17.116394,17.072041,0.482776,16.062317,17.796955,1.734638,...,26.072500,11.304584,1.00,0.640000,25.730000,8.657959,25.704063,11.171249,0.995,0.650000
3,0,World,4,3163,10.695886,10.434789,0.271945,9.915703,10.813854,0.898151,...,13.188750,1.786250,1.00,0.186667,12.976680,2.541891,12.789531,2.100781,0.955,0.248333
4,0,World,5,5222,9.152500,9.738646,0.434983,9.058959,10.305521,1.246562,...,2.136250,0.539583,0.24,0.053333,1.513203,-8.225443,2.016094,0.576406,0.225,0.065000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,3,Technology,16,7281,4.958855,4.098086,0.734436,2.895782,4.958855,2.063073,...,8.957500,4.635833,0.72,0.373333,8.722950,4.624863,8.547501,5.033438,0.700,0.416667
76,3,Technology,17,12559,4.935482,4.577303,0.258178,4.067435,4.935482,0.868047,...,3.100000,0.405417,0.44,0.053333,3.469800,-1.107503,2.443750,0.572135,0.335,0.073333
77,3,Technology,18,7152,4.660000,4.016875,0.475936,3.059999,4.660000,1.600000,...,0.000000,0.000000,0.00,0.000000,-26.318750,-30.335625,0.000000,0.000000,0.000,0.000000
78,3,Technology,19,12592,4.537498,4.100625,0.394584,3.491667,4.537498,1.045832,...,0.000000,0.000000,0.00,0.000000,-21.790939,-25.891563,0.000000,0.000000,0.000,0.000000



Originally top-ranked label-specific features flagged as LFH-sensitive by the simple multi-prefix rule:


...

Business: top-20 label-specific features


,label,label_word,label_specific_rank,feature_idx,baseline_label_specific_pre_gap,mean_label_specific_pre_gap_across_prefixes,std_label_specific_pre_gap_across_prefixes,min_label_specific_pre_gap_across_prefixes,max_label_specific_pre_gap_across_prefixes,range_label_specific_pre_gap_across_prefixes,...,baseline_mean_post_label,baseline_mean_post_other,baseline_activation_frequency_label,baseline_activation_frequency_other,mean_pre_label_across_prefixes,mean_pre_other_across_prefixes,mean_post_label_across_prefixes,mean_post_other_across_prefixes,activation_frequency_label_across_prefixes,activation_frequency_other_across_prefixes
0,2,Business,1,16288,18.021406,17.906029,0.819717,16.492188,19.204584,2.712397,...,13.192500,2.028750,0.88,0.200000,12.246855,-5.659173,12.898125,1.699375,0.810,0.176667
1,2,Business,2,5333,13.143958,13.979466,0.715816,13.109062,15.095729,1.986667,...,4.641250,1.220000,0.36,0.093333,0.689570,-13.289895,4.822187,1.472188,0.365,0.111667
2,2,Business,3,5349,10.461979,10.877018,0.254624,10.461979,11.209583,0.747604,...,5.710000,0.535833,0.48,0.066667,3.337148,-7.539869,5.371562,0.500156,0.480,0.060000
3,2,Business,4,1295,9.786667,8.961668,1.476517,6.840004,10.706665,3.866661,...,89.599998,79.813332,1.00,1.000000,88.279999,79.318336,88.279999,79.320000,1.000,0.998333
4,2,Business,5,16159,8.708261,8.988398,0.889292,7.797657,10.407084,2.609427,...,10.621250,5.459583,0.60,0.360000,7.467456,-1.520942,11.019062,5.763333,0.595,0.393333
5,2,Business,6,3678,7.642838,7.297445,0.532268,6.535228,8.058743,1.523516,...,4.456250,1.046667,0.40,0.120000,4.181824,-3.115621,4.601875,0.993542,0.465,0.123333
6,2,Business,7,3375,7.496718,9.034328,0.783126,7.496718,9.744219,2.247500,...,17.625000,13.141250,0.80,0.626667,18.132759,9.098431,18.420469,12.412292,0.870,0.600000
7,2,Business,8,2039,6.826914,6.845218,0.192631,6.487507,7.074707,0.587200,...,8.751250,1.240417,0.84,0.173333,9.311289,2.466071,8.868593,1.110000,0.865,0.155000
8,2,Business,9,15267,6.668333,6.343281,0.362414,5.701250,6.823334,1.122084,...,0.000000,0.000000,0.00,0.000000,-23.018282,-29.361563,0.000000,0.000000,0.000,0.000000
9,2,Business,10,14960,6.380482,6.098875,0.425689,5.367604,6.565729,1.198125,...,5.041250,0.901667,0.44,0.106667,4.904075,-1.194800,4.425312,0.654740,0.420,0.075000


Technology: top-20 label-specific features


,label,label_word,label_specific_rank,feature_idx,baseline_label_specific_pre_gap,mean_label_specific_pre_gap_across_prefixes,std_label_specific_pre_gap_across_prefixes,min_label_specific_pre_gap_across_prefixes,max_label_specific_pre_gap_across_prefixes,range_label_specific_pre_gap_across_prefixes,...,baseline_mean_post_label,baseline_mean_post_other,baseline_activation_frequency_label,baseline_activation_frequency_other,mean_pre_label_across_prefixes,mean_pre_other_across_prefixes,mean_post_label_across_prefixes,mean_post_other_across_prefixes,activation_frequency_label_across_prefixes,activation_frequency_other_across_prefixes
0,3,Technology,1,1831,18.446405,17.769663,0.399834,17.148815,18.446405,1.297590,...,5.548750,0.429583,0.40,0.026667,-0.809346,-18.579010,5.779375,0.452917,0.410,0.028333
1,3,Technology,2,3442,14.010313,13.757292,0.361856,13.073854,14.207812,1.133958,...,6.460000,1.104167,0.52,0.106667,6.358945,-7.398346,6.702500,1.107708,0.550,0.105000
2,3,Technology,3,16159,12.843471,12.190546,0.857285,10.102630,12.843471,2.740841,...,10.512500,5.495833,0.68,0.333333,9.869067,-2.321479,10.420626,5.962812,0.675,0.366667
3,3,Technology,4,7700,11.832865,12.011211,0.254297,11.573385,12.343229,0.769844,...,13.573750,2.967917,0.88,0.253333,14.510704,2.499492,14.463125,3.766719,0.905,0.330000
4,3,Technology,5,9247,11.271146,10.950583,0.705674,10.012917,12.210990,2.198073,...,6.308750,1.355000,0.40,0.133333,3.515400,-7.435183,6.415000,1.494948,0.455,0.146667
5,3,Technology,6,8875,9.317318,7.984427,0.650001,7.155416,9.317318,2.161901,...,6.562500,0.841667,0.60,0.106667,5.695752,-2.288675,6.142500,0.986354,0.535,0.115000
6,3,Technology,7,13811,8.518959,8.362705,0.487091,7.628333,9.117161,1.488828,...,1.492500,0.000000,0.20,0.000000,-0.066436,-8.429140,1.486875,0.011146,0.195,0.001667
7,3,Technology,8,14601,8.241979,7.082448,0.530843,6.416979,8.241979,1.824999,...,2.027500,0.241667,0.16,0.026667,-4.491601,-11.574049,1.840312,0.373490,0.185,0.048333
8,3,Technology,9,10070,7.406666,7.053958,0.599918,5.976669,7.993336,2.016666,...,0.000000,0.000000,0.00,0.000000,-40.016876,-47.070831,0.000000,0.000000,0.000,0.000000
9,3,Technology,10,10303,7.059792,6.406273,0.431194,5.728463,7.060521,1.332058,...,4.393750,0.932917,0.48,0.106667,3.391778,-3.014495,4.450937,0.876510,0.485,0.095000


Union of selected features across labels: 72 unique features.


## 5A. Rank table for baseline top-20 label-specific features across P2–P8

This diagnostic keeps the 80 features selected from the ordinary baseline prefix `P1` and reports where each one ranks under the alternative final-demo prefixes `P2` through `P8`. Rank `1` means the most label-specific feature for that label under that prefix; larger ranks indicate that the feature becomes less label-specific under that prefix.

In [7]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

In [8]:

# ============================================================
# 5A. Rank table: baseline top-20 label-specific features in P2..P8
# ============================================================

# This cell assumes Section 5 has already run and created:
#   label_specific_stability_df
#   label_variant_ranks
#   label_specific_pre_gap
#   label_top_features
#
# Interpretation:
#   P1 is the ordinary single-prefix baseline used to select top-20 features.
#   P2..P8 keep the same queries and demos 1..7, but replace only demo 8.
#   Rank 1 = most label-specific feature for that label under that prefix.

required_for_rank_table = [
    "label_specific_stability_df",
    "label_variant_ranks",
    "label_specific_pre_gap",
    "label_top_features",
]
missing = [name for name in required_for_rank_table if name not in globals()]
if missing:
    raise RuntimeError(
        "Run Sections 4 and 5 before this rank-table diagnostic. "
        f"Missing variables: {missing}"
    )

rank_rows = []
for _, row in label_specific_stability_df.iterrows():
    lab = int(row["label"])
    feat = int(row["feature_idx"])
    baseline_rank = int(row["label_specific_rank"])

    ranks = [int(label_variant_ranks[lab, v, feat]) for v in range(N_PREFIX_VARIANTS)]
    gaps = [float(label_specific_pre_gap[lab, v, feat]) for v in range(N_PREFIX_VARIANTS)]

    additional_ranks = ranks[1:]
    additional_gaps = gaps[1:]
    worst_additional_idx = int(np.argmax(additional_ranks)) + 1  # convert additional slot to variant index v
    best_additional_idx = int(np.argmin(additional_ranks)) + 1

    out = {
        "label_word": row["label_word"],
        "P1_baseline_rank": baseline_rank,
        "feature_idx": feat,
        "feature": f"L{TARGET_LAYER}-{feat}",
        "stable_by_rule": bool(row["stable_by_rule"]),
        "retention_rate_topK": float(row[f"retention_rate_top_{RETENTION_TOP_K}"]),
        "min_over_baseline": float(row["min_over_baseline"]),
        "baseline_gap_P1": float(gaps[0]),
        "worst_rank_P2_to_P8": int(max(additional_ranks)),
        "worst_prefix_P2_to_P8": f"P{worst_additional_idx + 1}",
        "best_rank_P2_to_P8": int(min(additional_ranks)),
        "best_prefix_P2_to_P8": f"P{best_additional_idx + 1}",
        f"n_P2_to_P8_in_top{RETENTION_TOP_K}": int(sum(r <= RETENTION_TOP_K for r in additional_ranks)),
    }

    # Add exact ranks and gaps for every prefix. P1 is included for context;
    # P2..P8 are the requested alternative-prefix ranks.
    for v in range(N_PREFIX_VARIANTS):
        out[f"P{v + 1}_rank"] = int(ranks[v])
    for v in range(N_PREFIX_VARIANTS):
        out[f"P{v + 1}_gap"] = float(gaps[v])

    rank_rows.append(out)

# Main table: one row per baseline-selected top-20 label-specific feature.
top20_label_specific_rank_by_prefix_df = pd.DataFrame(rank_rows)

rank_display_cols = (
    [
        "label_word",
        "P1_baseline_rank",
        "feature",
        "feature_idx",
        "stable_by_rule",
        "retention_rate_topK",
        "min_over_baseline",
        "baseline_gap_P1",
        "worst_rank_P2_to_P8",
        "worst_prefix_P2_to_P8",
        "best_rank_P2_to_P8",
        "best_prefix_P2_to_P8",
        f"n_P2_to_P8_in_top{RETENTION_TOP_K}",
    ]
    + [f"P{v + 1}_rank" for v in range(N_PREFIX_VARIANTS)]
)

# Requested table: rank under P2..P8 for each P1 top-20 label-specific feature.
print("=" * 100)
print(f"Ranks of P1 top-{LABEL_SPECIFIC_TOP_N} label-specific features under P2..P{N_PREFIX_VARIANTS}")
print("Rank 1 = most label-specific under that prefix. Larger ranks indicate loss of label specificity.")
print("=" * 100)
display(
    top20_label_specific_rank_by_prefix_df[rank_display_cols]
    .sort_values(["label_word", "P1_baseline_rank"])
    .reset_index(drop=True)
)

# Per-label compact tables, easier to copy into a rebuttal appendix.
for lab in range(NUM_LABELS):
    label_word = label_to_words_agnews[lab]
    print("=" * 100)
    print(f"{label_word}: P1 top-{LABEL_SPECIFIC_TOP_N} features and their ranks in P2..P{N_PREFIX_VARIANTS}")
    cols = (
        ["P1_baseline_rank", "feature", "stable_by_rule", "worst_rank_P2_to_P8", "worst_prefix_P2_to_P8"]
        + [f"P{v + 1}_rank" for v in range(1, N_PREFIX_VARIANTS)]
    )
    display(
        top20_label_specific_rank_by_prefix_df
        .loc[top20_label_specific_rank_by_prefix_df["label_word"] == label_word, cols]
        .sort_values("P1_baseline_rank")
        .reset_index(drop=True)
    )

# Worst rank collapses across all labels. This is useful for illustrating the single-prefix risk.
rank_collapse_examples_df = (
    top20_label_specific_rank_by_prefix_df
    .assign(rank_collapse=lambda df: df["worst_rank_P2_to_P8"] - df["P1_baseline_rank"])
    .sort_values("rank_collapse", ascending=False)
    .reset_index(drop=True)
)

print("=" * 100)
print("Largest rank-collapse examples among the P1 top-20 label-specific features")
print("=" * 100)
display(
    rank_collapse_examples_df[
        [
            "label_word", "P1_baseline_rank", "feature", "stable_by_rule",
            "worst_prefix_P2_to_P8", "worst_rank_P2_to_P8", "rank_collapse",
        ]
        + [f"P{v + 1}_rank" for v in range(N_PREFIX_VARIANTS)]
    ].head(20)
)

# Optional Plotly heatmap of log10(rank). This makes large rank collapses visible without saturating the scale.
def plot_top20_rank_heatmap_by_label(lab: int):
    label_word = label_to_words_agnews[int(lab)]
    sub = (
        top20_label_specific_rank_by_prefix_df
        .loc[top20_label_specific_rank_by_prefix_df["label_word"] == label_word]
        .sort_values("P1_baseline_rank")
        .reset_index(drop=True)
    )
    rank_cols = [f"P{v + 1}_rank" for v in range(N_PREFIX_VARIANTS)]
    ranks = sub[rank_cols].to_numpy(dtype=np.float64)
    log_ranks = np.log10(ranks)

    fig = go.Figure(
        data=go.Heatmap(
            z=log_ranks,
            x=[f"P{v + 1}" for v in range(N_PREFIX_VARIANTS)],
            y=[f"{r['feature']} (P1 rank {int(r['P1_baseline_rank'])})" for _, r in sub.iterrows()],
            colorscale="Blues",
            customdata=ranks,
            colorbar=dict(title="log10(rank)"),
            hovertemplate=(
                "feature=%{y}<br>prefix=%{x}<br>"
                "rank=%{customdata:.0f}<br>log10(rank)=%{z:.3f}<extra></extra>"
            ),
        )
    )
    fig.update_layout(
        title=f"{label_word}: ranks of P1 top-{LABEL_SPECIFIC_TOP_N} label-specific features across P1..P{N_PREFIX_VARIANTS}",
        xaxis_title="Prefix variant",
        yaxis_title="Feature selected from P1 baseline",
        height=max(450, 26 * len(sub) + 180),
        width=max(850, 90 * N_PREFIX_VARIANTS + 350),
    )
    fig.show()

for lab in range(NUM_LABELS):
    plot_top20_rank_heatmap_by_label(lab)


Ranks of P1 top-20 label-specific features under P2..P8
Rank 1 = most label-specific under that prefix. Larger ranks indicate loss of label specificity.


,label_word,P1_baseline_rank,feature,feature_idx,stable_by_rule,retention_rate_topK,min_over_baseline,baseline_gap_P1,worst_rank_P2_to_P8,worst_prefix_P2_to_P8,best_rank_P2_to_P8,best_prefix_P2_to_P8,n_P2_to_P8_in_top20,P1_rank,P2_rank,P3_rank,P4_rank,P5_rank,P6_rank,P7_rank,P8_rank
0,Business,1,L20-16288,16288,True,1.000,0.915144,18.021406,1,P2,1,P2,7,1,1,1,1,1,1,1,1
1,Business,2,L20-5333,5333,True,1.000,0.997345,13.143958,2,P2,2,P2,7,2,2,2,2,2,2,2,2
2,Business,3,L20-5349,5349,True,1.000,1.000000,10.461979,3,P2,3,P2,7,3,3,3,3,3,3,3,3
3,Business,4,L20-1295,1295,True,1.000,0.698910,9.786667,8,P8,4,P3,7,4,7,4,5,5,4,6,8
4,Business,5,L20-16159,16159,True,1.000,0.895432,8.708261,6,P5,4,P4,7,5,5,5,4,6,6,5,4
5,Business,6,L20-3678,3678,True,1.000,0.855079,7.642838,8,P4,6,P2,7,6,6,7,8,7,8,7,6
6,Business,7,L20-3375,3375,True,1.000,1.000000,7.496718,6,P3,4,P2,7,7,4,6,6,4,5,4,5
7,Business,8,L20-2039,2039,True,1.000,0.950284,6.826914,8,P2,7,P4,7,8,8,8,7,8,7,8,7
8,Business,9,L20-15267,15267,True,1.000,0.854974,6.668333,13,P2,9,P3,7,9,13,9,9,9,10,9,10
9,Business,10,L20-14960,14960,True,1.000,0.841254,6.380482,11,P3,9,P2,7,10,9,11,10,10,9,10,9


World: P1 top-20 features and their ranks in P2..P8


,P1_baseline_rank,feature,stable_by_rule,worst_rank_P2_to_P8,worst_prefix_P2_to_P8,P2_rank,P3_rank,P4_rank,P5_rank,P6_rank,P7_rank,P8_rank
0,1,L20-13844,True,1,P2,1,1,1,1,1,1,1
1,2,L20-3994,True,2,P2,2,2,2,2,2,2,2
2,3,L20-14744,True,3,P2,3,3,3,3,3,3,3
3,4,L20-3163,True,6,P4,4,4,6,4,4,4,4
4,5,L20-5222,True,6,P3,5,6,4,6,5,5,5
5,6,L20-8901,True,6,P2,6,5,5,5,6,6,6
6,7,L20-406,True,8,P6,7,7,7,7,8,8,7
7,8,L20-1728,True,9,P2,9,9,8,8,7,7,8
8,9,L20-9069,True,10,P8,8,8,9,9,9,9,10
9,10,L20-5743,True,15,P6,12,10,10,11,15,12,11


Sports: P1 top-20 features and their ranks in P2..P8


...

,P1_baseline_rank,feature,stable_by_rule,worst_rank_P2_to_P8,worst_prefix_P2_to_P8,P2_rank,P3_rank,P4_rank,P5_rank,P6_rank,P7_rank,P8_rank
0,1,L20-16288,True,1,P2,1,1,1,1,1,1,1
1,2,L20-5333,True,2,P2,2,2,2,2,2,2,2
2,3,L20-5349,True,3,P2,3,3,3,3,3,3,3
3,4,L20-1295,True,8,P8,7,4,5,5,4,6,8
4,5,L20-16159,True,6,P5,5,5,4,6,6,5,4
5,6,L20-3678,True,8,P4,6,7,8,7,8,7,6
6,7,L20-3375,True,6,P3,4,6,6,4,5,4,5
7,8,L20-2039,True,8,P2,8,8,7,8,7,8,7
8,9,L20-15267,True,13,P2,13,9,9,9,10,9,10
9,10,L20-14960,True,11,P3,9,11,10,10,9,10,9


Technology: P1 top-20 features and their ranks in P2..P8


,P1_baseline_rank,feature,stable_by_rule,worst_rank_P2_to_P8,worst_prefix_P2_to_P8,P2_rank,P3_rank,P4_rank,P5_rank,P6_rank,P7_rank,P8_rank
0,1,L20-1831,True,1,P2,1,1,1,1,1,1,1
1,2,L20-3442,True,2,P2,2,2,2,2,2,2,2
2,3,L20-16159,True,5,P6,4,3,3,3,5,3,3
3,4,L20-7700,True,5,P3,3,5,4,4,3,4,4
4,5,L20-9247,True,5,P2,5,4,5,5,4,5,5
5,6,L20-8875,True,8,P3,6,8,7,7,7,7,7
6,7,L20-13811,True,7,P2,7,6,6,6,6,6,6
7,8,L20-14601,True,9,P2,9,7,9,8,9,9,9
8,9,L20-10070,True,10,P3,8,10,8,10,8,8,8
9,10,L20-10303,True,10,P2,10,9,10,9,10,10,10


Largest rank-collapse examples among the P1 top-20 label-specific features


,label_word,P1_baseline_rank,feature,stable_by_rule,worst_prefix_P2_to_P8,worst_rank_P2_to_P8,rank_collapse,P1_rank,P2_rank,P3_rank,P4_rank,P5_rank,P6_rank,P7_rank,P8_rank
0,World,11,L20-1295,False,P4,43,32,11,11,14,43,10,10,13,9
1,Technology,16,L20-7281,False,P2,48,32,16,48,22,38,19,15,17,20
2,Sports,18,L20-7502,False,P7,47,29,18,21,41,37,26,23,47,20
3,Sports,19,L20-2572,False,P7,46,27,19,19,28,33,19,14,46,16
4,Technology,18,L20-7152,False,P3,43,25,18,22,43,23,25,20,25,17
5,Technology,20,L20-6880,False,P2,39,19,20,39,32,16,20,21,27,35
6,World,12,L20-6211,False,P7,25,13,12,17,17,17,15,16,25,12
7,Business,18,L20-9009,False,P4,29,11,18,21,25,29,17,21,22,19
8,Technology,19,L20-12592,False,P3,29,10,19,18,29,28,24,18,23,18
9,Technology,12,L20-8551,False,P4,21,9,12,14,13,21,12,13,14,14


In [9]:

# ============================================================
# 6. Plotly diagnostics and heatmaps
# ============================================================

summary = {
    "model": MODEL_SHORT_NAME,
    "target_layer": TARGET_LAYER,
    "n_queries": N_QUERY_TOTAL,
    "n_prefix_variants": N_PREFIX_VARIANTS,
    "label_specific_top_n_per_label": LABEL_SPECIFIC_TOP_N,
    "retention_top_k": RETENTION_TOP_K,
    "stable_min_retention": STABLE_MIN_RETENTION,
    "stable_min_baseline_ratio": STABLE_MIN_BASELINE_RATIO,
    "query_analysis_positions": query_analysis_positions,
    "stable_count": int(label_specific_stability_df["stable_by_rule"].sum()),
    "sensitive_count": int((~label_specific_stability_df["stable_by_rule"]).sum()),
}
print(json.dumps(summary, indent=2))

# Per-variant final-demo label distribution. This checks whether variant index itself is label-confounded.
variant_label_counts = pd.crosstab(meta_df["variant_idx"], meta_df["final_demo_label_word"])
print("Final-demo label counts by variant index:")
display(variant_label_counts)

fig_counts = px.imshow(
    variant_label_counts,
    text_auto=True,
    aspect="auto",
    title="Final-demo label counts by prefix-variant index",
    labels=dict(x="Final-demo label", y="Variant index", color="count"),
)
fig_counts.show()


def plot_label_specific_heatmap(lab: int, normalize_to_baseline: bool = False):
    """Plot top label-specific features for one label across final-demo variants."""
    feature_ids = label_top_features[int(lab)]
    mat = label_specific_pre_gap[int(lab), :, feature_ids].T  # [features, variants]
    if normalize_to_baseline:
        denom = np.abs(mat[:, [0]]) + 1e-12
        mat_to_plot = mat / denom
        colorbar_title = "pre-gap / |baseline gap|"
        title_suffix = "ratio to baseline"
    else:
        mat_to_plot = mat
        colorbar_title = "label-specific preactivation gap"
        title_suffix = "raw gap"

    y_labels = [f"L{TARGET_LAYER}-{int(f)}" for f in feature_ids]
    x_labels = [f"v{v}" for v in range(N_PREFIX_VARIANTS)]
    fig = go.Figure(
        data=go.Heatmap(
            z=mat_to_plot,
            x=x_labels,
            y=y_labels,
            colorscale="RdBu",
            reversescale=True,
            zmid=0.0,
            colorbar=dict(title=colorbar_title),
            hovertemplate=(
                "feature=%{y}<br>variant=%{x}<br>"
                + colorbar_title
                + "=%{z:.4f}<extra></extra>"
            ),
        )
    )
    fig.update_layout(
        title=f"{label_to_words_agnews[int(lab)]}: top-{LABEL_SPECIFIC_TOP_N} label-specific features across final-demo variants ({title_suffix})",
        xaxis_title="Final-demo prefix variant",
        yaxis_title="SAE feature",
        height=max(450, 26 * len(feature_ids) + 180),
        width=max(850, 80 * N_PREFIX_VARIANTS + 350),
    )
    fig.show()


for lab in range(NUM_LABELS):
    plot_label_specific_heatmap(lab, normalize_to_baseline=False)

# Optional normalized heatmaps. These are useful for visually identifying LFH-sensitive features.
for lab in range(NUM_LABELS):
    plot_label_specific_heatmap(lab, normalize_to_baseline=True)

# Retention plot for all selected label-specific features.
fig_retention = px.scatter(
    label_specific_stability_df,
    x="label_specific_rank",
    y=f"retention_rate_top_{RETENTION_TOP_K}",
    color="label_word",
    symbol="stable_by_rule",
    hover_data=[
        "feature_idx",
        "baseline_label_specific_pre_gap",
        "min_over_baseline",
        "worst_variant_rank",
        "best_variant_rank",
    ],
    title=f"Single-prefix label-specific rank versus multi-prefix retention in top-{RETENTION_TOP_K}",
    labels={
        "label_specific_rank": "Original single-prefix label-specific rank",
        f"retention_rate_top_{RETENTION_TOP_K}": f"Retention rate in top-{RETENTION_TOP_K}",
        "label_word": "Gold label",
    },
)
fig_retention.add_hline(y=STABLE_MIN_RETENTION, line_dash="dash")
fig_retention.update_layout(height=500, width=900)
fig_retention.show()

# Gap range plot: how much the label-specific score moves when only the final demo changes.
fig_range = px.scatter(
    label_specific_stability_df,
    x="baseline_label_specific_pre_gap",
    y="range_label_specific_pre_gap_across_prefixes",
    color="label_word",
    symbol="stable_by_rule",
    hover_data=["feature_idx", "label_specific_rank", "min_over_baseline", f"retention_rate_top_{RETENTION_TOP_K}"],
    title="Label-specific preactivation gap instability under final-demo variation",
    labels={
        "baseline_label_specific_pre_gap": "Baseline label-specific preactivation gap",
        "range_label_specific_pre_gap_across_prefixes": "Range of gap across prefix variants",
        "label_word": "Gold label",
    },
)
fig_range.update_layout(height=500, width=900)
fig_range.show()


{
  "model": "Gemma2-9B-IT",
  "target_layer": 20,
  "n_queries": 100,
  "n_prefix_variants": 8,
  "label_specific_top_n_per_label": 20,
  "retention_top_k": 20,
  "stable_min_retention": 0.9,
  "stable_min_baseline_ratio": 0.5,
  "query_analysis_positions": [
    -2
  ],
  "stable_count": 61,
  "sensitive_count": 19
}
Final-demo label counts by variant index:


final_demo_label_word,Business,Sports,Technology,World
variant_idx,,,,
0,24,19,31,26
1,29,26,27,18
2,29,23,20,28
3,23,26,27,24
4,24,26,24,26
5,29,29,23,19
6,23,29,22,26
7,19,22,26,33



# 8. Causal validation: are multi-prefix-stable label features more causally relevant?

This section tests the hypothesis behind the LFH-aware stability filter:

- **Stable group:** label-specific features that remain label-specific across final-demo variants.
- **LFH-sensitive group:** label-specific features that looked strong under the original single-prefix prompt but failed the multi-prefix stability rule.

For every selected `(label, feature)` pair, we evaluate prompts whose gold query label is that label and intervene at the same query-token positions used for feature measurement.

Two interventions are run:

1. **Active-feature ablation:** when the feature is active at an edited token, subtract its current SAE reconstruction contribution:
   `hidden <- hidden - activation_f * W_dec[f]`.
2. **Feature amplification:** add the feature decoder direction with the current value of `CAUSAL_AMPLIFICATION_ALPHA`:
   `hidden <- hidden + CAUSAL_AMPLIFICATION_ALPHA * W_dec[f]`.

The outcome is the model's next-token probability for the gold label, normalized over the four AGNews label tokens. The main quantities are:

- `ablation_decrease_p_label = baseline P(L) - ablated P(L)`; larger is stronger causal support for the label.
- `amplification_increase_p_label = amplified P(L) - baseline P(L)`; larger is stronger causal support for the label.


In [10]:

# ============================================================
# 8. Causal validation of stable versus LFH-sensitive features
# ============================================================

# Causal validation settings.
# By default this evaluates all 80 selected label-feature pairs: top-20 per AGNews label.
CAUSAL_MAX_PAIRS = None          # None = all selected pairs; set e.g. 8 for a quick smoke test
CAUSAL_BATCH_SIZE = BATCH_SIZE   # reduce if you hit memory limits
CAUSAL_AMPLIFICATION_ALPHA = 50.0  # coarse preliminary setting from the attached candidate notebook
CAUSAL_BOOTSTRAP_SAMPLES = 2000

# The causal outcome is P(label) normalized over the four label words. This is usually the
# most interpretable probability for a classification prompt.
CAUSAL_PROBABILITY_MODE = "label_conditional"  # fixed in this cell

print("CAUSAL_MAX_PAIRS:", CAUSAL_MAX_PAIRS)
print("CAUSAL_BATCH_SIZE:", CAUSAL_BATCH_SIZE)
print("CAUSAL_AMPLIFICATION_ALPHA:", CAUSAL_AMPLIFICATION_ALPHA)
print("CAUSAL_PROBABILITY_MODE:", CAUSAL_PROBABILITY_MODE)


def get_single_token_label_ids():
    """Return one tokenizer id per AGNews label word, with sanity checks."""
    label_ids = []
    rows = []
    for lab in range(NUM_LABELS):
        word = label_to_words_agnews[lab]
        ids = tokenizer(word, add_special_tokens=False)["input_ids"]
        rows.append({"label": lab, "label_word": word, "token_ids": ids, "n_tokens": len(ids)})
        if len(ids) != 1:
            raise ValueError(
                f"Label word {word!r} tokenized into {len(ids)} tokens: {ids}. "
                "This causal cell expects single-token labels."
            )
        label_ids.append(int(ids[0]))
    display(pd.DataFrame(rows))
    return label_ids


LABEL_TOKEN_IDS = get_single_token_label_ids()
LABEL_TOKEN_IDS_TENSOR = torch.tensor(LABEL_TOKEN_IDS, dtype=torch.long, device=get_model_input_device(model))


def label_conditional_probs_from_logits(logits: torch.Tensor) -> torch.Tensor:
    """
    Return P(label) normalized over the four AGNews label tokens.

    logits: [B, S, vocab]. We use logits at the final prompt token to score the next token.
    Returns: [B, NUM_LABELS].
    """
    label_token_ids = LABEL_TOKEN_IDS_TENSOR.to(logits.device)
    next_logits = logits[:, -1, :].float()
    label_logits = next_logits[:, label_token_ids]
    return torch.softmax(label_logits, dim=-1)


@torch.no_grad()
def collect_baseline_label_probs(prompts, batch_size: int = CAUSAL_BATCH_SIZE):
    """Collect baseline next-token P(label), normalized across the four label tokens, for every prompt."""
    model_device = get_model_input_device(model)
    probs = np.zeros((len(prompts), NUM_LABELS), dtype=np.float32)
    model.eval()

    for start in range(0, len(prompts), batch_size):
        end = min(start + batch_size, len(prompts))
        enc = tokenizer(
            prompts[start:end],
            padding=True,
            return_tensors="pt",
            add_special_tokens=False,
        )
        input_ids = enc["input_ids"].to(model_device)
        attention_mask = enc["attention_mask"].to(model_device)

        out = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)
        label_probs = label_conditional_probs_from_logits(out.logits)
        probs[start:end] = label_probs.detach().cpu().numpy()

        del out, input_ids, attention_mask, enc, label_probs
        print(f"Baseline probabilities: {end}/{len(prompts)}")

    return probs


baseline_label_probs_by_prompt = collect_baseline_label_probs(prompts)
print("baseline_label_probs_by_prompt:", baseline_label_probs_by_prompt.shape)


# Build the label-feature pairs to test. Each row corresponds to the same pair used in the
# stability table, so duplicated feature ids across labels are intentionally retained.
causal_pairs_df = label_specific_stability_df.copy().reset_index(drop=False).rename(columns={"index": "pair_id"})
if CAUSAL_MAX_PAIRS is not None:
    # Balanced quick mode: take from stable and sensitive groups separately when possible.
    stable_part = causal_pairs_df[causal_pairs_df["stable_by_rule"]].head(int(CAUSAL_MAX_PAIRS) // 2)
    sensitive_part = causal_pairs_df[~causal_pairs_df["stable_by_rule"]].head(int(CAUSAL_MAX_PAIRS) - len(stable_part))
    causal_pairs_df = pd.concat([stable_part, sensitive_part], ignore_index=True)

causal_pairs_df["causal_group"] = np.where(causal_pairs_df["stable_by_rule"], "multi-prefix stable", "LFH-sensitive")
print("Causal pairs to test:", len(causal_pairs_df))
display(causal_pairs_df[[
    "pair_id", "label_word", "feature_idx", "label_specific_rank", "stable_by_rule",
    "baseline_label_specific_pre_gap", f"retention_rate_top_{RETENTION_TOP_K}", "min_over_baseline", "worst_variant_rank"
]])


# Construct one intervention instance per (label-feature pair, prompt whose query label matches that label).
instance_rows = []
for _, pair in causal_pairs_df.iterrows():
    lab = int(pair["label"])
    feat = int(pair["feature_idx"])
    pair_id = int(pair["pair_id"])
    prompt_indices_for_label = meta_df.index[meta_df["query_label"].to_numpy(dtype=np.int64) == lab].to_numpy(dtype=np.int64)
    for prompt_idx in prompt_indices_for_label:
        meta = prompt_metadata[int(prompt_idx)]
        instance_rows.append({
            "pair_id": pair_id,
            "prompt_idx": int(prompt_idx),
            "query_local_idx": int(meta["query_local_idx"]),
            "variant_idx": int(meta["variant_idx"]),
            "label": lab,
            "label_word": label_to_words_agnews[lab],
            "feature_idx": feat,
        })

causal_instances_df = pd.DataFrame(instance_rows)
print("Causal intervention instances:", len(causal_instances_df))
display(causal_instances_df.head())


def _selected_padded_positions_for_batch(batch_meta, attention_mask, batch_seq_len: int):
    """Convert saved unpadded selected query positions into padded batch positions."""
    selected_positions_by_row = []
    for b, meta in enumerate(batch_meta):
        unpadded_len = int(attention_mask[b].sum().item())
        pad_shift = int(batch_seq_len - unpadded_len)
        positions = [pad_shift + int(pos) for pos in meta["selected_query_token_positions"]]
        positions = [pos for pos in positions if 0 <= pos < batch_seq_len]
        if not positions:
            raise RuntimeError("No valid selected token positions after padding adjustment.")
        selected_positions_by_row.append(positions)
    return selected_positions_by_row


@torch.no_grad()
def collect_intervened_gold_label_probs(instances_df: pd.DataFrame, mode: str, batch_size: int = CAUSAL_BATCH_SIZE):
    """
    Run feature-level interventions for prompt-feature instances.

    mode:
      - 'ablate_active': subtract current post-activation_f * W_dec[f] on selected query tokens.
      - 'amplify': add CAUSAL_AMPLIFICATION_ALPHA * W_dec[f] on selected query tokens.

    Returns: [n_instances] containing P(gold label) after intervention.
    """
    if mode not in {"ablate_active", "amplify"}:
        raise ValueError(f"Unknown intervention mode: {mode}")

    n = len(instances_df)
    gold_probs = np.zeros(n, dtype=np.float32)
    model_device = get_model_input_device(model)
    block = model.get_submodule(f"model.layers.{TARGET_LAYER}")
    model.eval()
    sae.eval()

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        batch_rows = instances_df.iloc[start:end]
        batch_prompt_indices = batch_rows["prompt_idx"].to_numpy(dtype=np.int64)
        batch_prompts = [prompts[int(i)] for i in batch_prompt_indices]
        batch_meta = [prompt_metadata[int(i)] for i in batch_prompt_indices]
        batch_features = batch_rows["feature_idx"].to_numpy(dtype=np.int64)
        batch_labels = batch_rows["label"].to_numpy(dtype=np.int64)

        enc = tokenizer(
            batch_prompts,
            padding=True,
            return_tensors="pt",
            add_special_tokens=False,
        )
        input_ids = enc["input_ids"].to(model_device)
        attention_mask = enc["attention_mask"].to(model_device)
        batch_seq_len = int(input_ids.shape[1])
        selected_positions_by_row = _selected_padded_positions_for_batch(batch_meta, attention_mask, batch_seq_len)

        def hook_fn(module, inputs, output):
            hidden = output[0] if isinstance(output, tuple) else output
            hidden_mod = hidden.clone()
            hidden_device = hidden_mod.device
            hidden_dtype = hidden_mod.dtype

            for b, feat in enumerate(batch_features):
                feat = int(feat)
                pos = selected_positions_by_row[b]
                x = hidden[b, pos, :]
                decoder_dir = sae.W_dec[feat].to(device=hidden_device, dtype=hidden_dtype)

                if mode == "ablate_active":
                    # Compute the current SAE post-activation and remove only the active contribution.
                    pre = sae_encode_pre_for_positions(hidden[b], pos)
                    post = sae.activation_fn(pre.to(dtype=sae.dtype)).float()
                    coeff = post[:, feat].to(device=hidden_device, dtype=hidden_dtype)
                    hidden_mod[b, pos, :] = hidden_mod[b, pos, :] - coeff[:, None] * decoder_dir[None, :]
                elif mode == "amplify":
                    hidden_mod[b, pos, :] = hidden_mod[b, pos, :] + float(CAUSAL_AMPLIFICATION_ALPHA) * decoder_dir[None, :]

            if isinstance(output, tuple):
                return (hidden_mod,) + output[1:]
            return hidden_mod

        handle = block.register_forward_hook(hook_fn)
        try:
            out = model(input_ids=input_ids, attention_mask=attention_mask, use_cache=False)
        finally:
            handle.remove()

        label_probs = label_conditional_probs_from_logits(out.logits).detach().cpu().numpy()
        gold_probs[start:end] = label_probs[np.arange(end - start), batch_labels]

        del out, input_ids, attention_mask, enc, label_probs
        print(f"{mode}: {end}/{n}")

    return gold_probs


# Baseline summaries per pair using the same prompt subset as each intervention.
baseline_rows = []
for _, pair in causal_pairs_df.iterrows():
    pair_id = int(pair["pair_id"])
    lab = int(pair["label"])
    feat = int(pair["feature_idx"])
    prompt_indices = causal_instances_df.loc[causal_instances_df["pair_id"] == pair_id, "prompt_idx"].to_numpy(dtype=np.int64)

    baseline_gold = baseline_label_probs_by_prompt[prompt_indices, lab]
    feature_freq = freq_by_prompt[prompt_indices, feat]
    feature_post = post_by_prompt[prompt_indices, feat]
    feature_pre = pre_by_prompt[prompt_indices, feat]

    baseline_rows.append({
        "pair_id": pair_id,
        "baseline_mean_p_label": float(np.mean(baseline_gold)),
        "baseline_median_p_label": float(np.median(baseline_gold)),
        "n_prompts_for_label": int(len(prompt_indices)),
        "feature_active_prompt_fraction": float(np.mean(feature_freq > 0)),
        "mean_feature_preactivation_on_gold_label_prompts": float(np.mean(feature_pre)),
        "mean_feature_postactivation_on_gold_label_prompts": float(np.mean(feature_post)),
    })

baseline_pair_df = pd.DataFrame(baseline_rows)

# Run the two causal interventions.
ablation_gold_probs = collect_intervened_gold_label_probs(causal_instances_df, mode="ablate_active")
amplified_gold_probs = collect_intervened_gold_label_probs(causal_instances_df, mode="amplify")

causal_instances_df = causal_instances_df.copy()
causal_instances_df["baseline_p_label"] = baseline_label_probs_by_prompt[
    causal_instances_df["prompt_idx"].to_numpy(dtype=np.int64),
    causal_instances_df["label"].to_numpy(dtype=np.int64),
]
causal_instances_df["ablated_p_label"] = ablation_gold_probs
causal_instances_df["amplified_p_label"] = amplified_gold_probs
causal_instances_df["ablation_decrease_p_label"] = causal_instances_df["baseline_p_label"] - causal_instances_df["ablated_p_label"]
causal_instances_df["amplification_increase_p_label"] = causal_instances_df["amplified_p_label"] - causal_instances_df["baseline_p_label"]

# Aggregate intervention effects per label-feature pair.
causal_pair_effects_df = (
    causal_instances_df
    .groupby("pair_id")
    .agg(
        ablated_mean_p_label=("ablated_p_label", "mean"),
        amplified_mean_p_label=("amplified_p_label", "mean"),
        mean_ablation_decrease_p_label=("ablation_decrease_p_label", "mean"),
        median_ablation_decrease_p_label=("ablation_decrease_p_label", "median"),
        mean_amplification_increase_p_label=("amplification_increase_p_label", "mean"),
        median_amplification_increase_p_label=("amplification_increase_p_label", "median"),
        frac_prompts_ablation_decreases_p_label=("ablation_decrease_p_label", lambda x: float(np.mean(np.asarray(x) > 0))),
        frac_prompts_amplification_increases_p_label=("amplification_increase_p_label", lambda x: float(np.mean(np.asarray(x) > 0))),
    )
    .reset_index()
)

causal_validation_df = (
    causal_pairs_df
    .merge(baseline_pair_df, on="pair_id", how="left")
    .merge(causal_pair_effects_df, on="pair_id", how="left")
)

# Convenience columns for reading.
causal_validation_df["ablation_decrease_pp"] = 100.0 * causal_validation_df["mean_ablation_decrease_p_label"]
causal_validation_df["amplification_increase_pp"] = 100.0 * causal_validation_df["mean_amplification_increase_p_label"]
causal_validation_df["baseline_p_label_pct"] = 100.0 * causal_validation_df["baseline_mean_p_label"]
causal_validation_df["ablated_p_label_pct"] = 100.0 * causal_validation_df["ablated_mean_p_label"]
causal_validation_df["amplified_p_label_pct"] = 100.0 * causal_validation_df["amplified_mean_p_label"]

print("Causal validation per label-feature pair:")
display(causal_validation_df[[
    "label_word", "feature_idx", "label_specific_rank", "stable_by_rule", "causal_group",
    "baseline_label_specific_pre_gap", f"retention_rate_top_{RETENTION_TOP_K}", "min_over_baseline",
    "feature_active_prompt_fraction", "baseline_p_label_pct", "ablated_p_label_pct", "amplified_p_label_pct",
    "ablation_decrease_pp", "amplification_increase_pp",
    "frac_prompts_ablation_decreases_p_label", "frac_prompts_amplification_increases_p_label",
]].sort_values(["stable_by_rule", "label_word", "label_specific_rank"], ascending=[False, True, True]))


def bootstrap_mean_ci(values, n_boot: int = CAUSAL_BOOTSTRAP_SAMPLES, seed: int = seed, ci: float = 0.95):
    vals = np.asarray(values, dtype=np.float64)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return np.nan, np.nan, np.nan
    mean = float(vals.mean())
    if vals.size == 1 or n_boot <= 0:
        return mean, np.nan, np.nan
    rng_boot = np.random.default_rng(seed)
    draws = rng_boot.choice(vals, size=(int(n_boot), vals.size), replace=True).mean(axis=1)
    lo = float(np.quantile(draws, (1.0 - ci) / 2.0))
    hi = float(np.quantile(draws, 1.0 - (1.0 - ci) / 2.0))
    return mean, lo, hi


group_rows = []
for stable_value, group_name in [(True, "multi-prefix stable"), (False, "LFH-sensitive")]:
    g = causal_validation_df[causal_validation_df["stable_by_rule"] == stable_value]
    row = {
        "causal_group": group_name,
        "n_label_feature_pairs": int(len(g)),
        "mean_active_prompt_fraction": float(g["feature_active_prompt_fraction"].mean()) if len(g) else np.nan,
        "median_active_prompt_fraction": float(g["feature_active_prompt_fraction"].median()) if len(g) else np.nan,
    }
    for metric, label in [
        ("mean_ablation_decrease_p_label", "ablation_decrease"),
        ("mean_amplification_increase_p_label", "amplification_increase"),
    ]:
        mean, lo, hi = bootstrap_mean_ci(g[metric].to_numpy(), seed=seed + (1 if stable_value else 2))
        row[f"mean_{label}"] = mean
        row[f"mean_{label}_ci_low"] = lo
        row[f"mean_{label}_ci_high"] = hi
        row[f"mean_{label}_pp"] = 100.0 * mean if np.isfinite(mean) else np.nan
        row[f"mean_{label}_ci_low_pp"] = 100.0 * lo if np.isfinite(lo) else np.nan
        row[f"mean_{label}_ci_high_pp"] = 100.0 * hi if np.isfinite(hi) else np.nan
    group_rows.append(row)

causal_group_summary_df = pd.DataFrame(group_rows)
print("Stable versus LFH-sensitive causal summary:")
display(causal_group_summary_df)

# Label-level group summary, useful for finding whether one label dominates the comparison.
label_group_summary_df = (
    causal_validation_df
    .groupby(["label_word", "causal_group"])
    .agg(
        n_label_feature_pairs=("feature_idx", "count"),
        mean_ablation_decrease_pp=("ablation_decrease_pp", "mean"),
        mean_amplification_increase_pp=("amplification_increase_pp", "mean"),
        mean_active_prompt_fraction=("feature_active_prompt_fraction", "mean"),
    )
    .reset_index()
)
print("Per-label stable/sensitive causal summary:")
display(label_group_summary_df)

# Plotly comparison: ablation and amplification effects by stability group.
plot_df = causal_validation_df.copy()
plot_df["stability_group"] = np.where(plot_df["stable_by_rule"], "multi-prefix stable", "LFH-sensitive")

fig_ablation = px.box(
    plot_df,
    x="stability_group",
    y="ablation_decrease_pp",
    color="label_word",
    points="all",
    hover_data=["feature_idx", "label_specific_rank", "feature_active_prompt_fraction", f"retention_rate_top_{RETENTION_TOP_K}", "min_over_baseline"],
    title="Feature ablation: decrease in gold-label probability by multi-prefix stability group",
    labels={
        "stability_group": "Feature group",
        "ablation_decrease_pp": "Baseline P(L) - ablated P(L), percentage points",
        "label_word": "Gold label",
    },
)
fig_ablation.add_hline(y=0.0, line_dash="dash")
fig_ablation.update_layout(height=550, width=950)
fig_ablation.show()

fig_amplification = px.box(
    plot_df,
    x="stability_group",
    y="amplification_increase_pp",
    color="label_word",
    points="all",
    hover_data=["feature_idx", "label_specific_rank", "feature_active_prompt_fraction", f"retention_rate_top_{RETENTION_TOP_K}", "min_over_baseline"],
    title=f"Feature amplification (alpha={CAUSAL_AMPLIFICATION_ALPHA:g}): increase in gold-label probability by stability group",
    labels={
        "stability_group": "Feature group",
        "amplification_increase_pp": "Amplified P(L) - baseline P(L), percentage points",
        "label_word": "Gold label",
    },
)
fig_amplification.add_hline(y=0.0, line_dash="dash")
fig_amplification.update_layout(height=550, width=950)
fig_amplification.show()

# Scatter: stability score versus causal effect. Ideally, more stable features have stronger causal effects.
fig_scatter = px.scatter(
    plot_df,
    x=f"retention_rate_top_{RETENTION_TOP_K}",
    y="ablation_decrease_pp",
    color="label_word",
    symbol="stable_by_rule",
    size="feature_active_prompt_fraction",
    hover_data=["feature_idx", "label_specific_rank", "amplification_increase_pp", "min_over_baseline", "worst_variant_rank"],
    title="Does multi-prefix retention predict feature-ablation causal relevance?",
    labels={
        f"retention_rate_top_{RETENTION_TOP_K}": f"Retention rate in top-{RETENTION_TOP_K}",
        "ablation_decrease_pp": "Ablation decrease in P(L), percentage points",
        "label_word": "Gold label",
    },
)
fig_scatter.add_hline(y=0.0, line_dash="dash")
fig_scatter.add_vline(x=STABLE_MIN_RETENTION, line_dash="dash")
fig_scatter.update_layout(height=550, width=950)
fig_scatter.show()

# Compact interpretation helper.
stable_mean_ablate = causal_group_summary_df.loc[causal_group_summary_df["causal_group"] == "multi-prefix stable", "mean_ablation_decrease_pp"].iloc[0]
sens_mean_ablate = causal_group_summary_df.loc[causal_group_summary_df["causal_group"] == "LFH-sensitive", "mean_ablation_decrease_pp"].iloc[0]
stable_mean_amp = causal_group_summary_df.loc[causal_group_summary_df["causal_group"] == "multi-prefix stable", "mean_amplification_increase_pp"].iloc[0]
sens_mean_amp = causal_group_summary_df.loc[causal_group_summary_df["causal_group"] == "LFH-sensitive", "mean_amplification_increase_pp"].iloc[0]

print("=" * 100)
print("Quick readout")
print("=" * 100)
print(f"Mean ablation decrease, stable group:      {stable_mean_ablate:.4f} percentage points")
print(f"Mean ablation decrease, LFH-sensitive:     {sens_mean_ablate:.4f} percentage points")
print(f"Mean amplification increase, stable group: {stable_mean_amp:.4f} percentage points")
print(f"Mean amplification increase, sensitive:    {sens_mean_amp:.4f} percentage points")
print()
if stable_mean_ablate > sens_mean_ablate and stable_mean_amp > sens_mean_amp:
    print("Pattern matches the ideal validation: multi-prefix-stable features are more causally relevant on both tests.")
elif stable_mean_ablate > sens_mean_ablate or stable_mean_amp > sens_mean_amp:
    print("Pattern is partially supportive: stable features are stronger on one causal test but not both.")
else:
    print("Pattern is not supportive under this run/settings; inspect label-wise summaries and active fractions.")


CAUSAL_MAX_PAIRS: None
CAUSAL_BATCH_SIZE: 2
CAUSAL_AMPLIFICATION_ALPHA: 50.0
CAUSAL_PROBABILITY_MODE: label_conditional


,label,label_word,token_ids,n_tokens
0,0,World,[10772],1
1,1,Sports,[27093],1
2,2,Business,[14103],1
3,3,Technology,[39364],1


Baseline probabilities: 2/800
Baseline probabilities: 4/800
Baseline probabilities: 6/800
Baseline probabilities: 8/800
Baseline probabilities: 10/800
Baseline probabilities: 12/800
Baseline probabilities: 14/800
Baseline probabilities: 16/800
Baseline probabilities: 18/800
Baseline probabilities: 20/800
Baseline probabilities: 22/800
Baseline probabilities: 24/800
Baseline probabilities: 26/800
Baseline probabilities: 28/800
Baseline probabilities: 30/800
Baseline probabilities: 32/800
Baseline probabilities: 34/800
Baseline probabilities: 36/800
Baseline probabilities: 38/800
Baseline probabilities: 40/800
Baseline probabilities: 42/800
Baseline probabilities: 44/800
Baseline probabilities: 46/800
Baseline probabilities: 48/800
Baseline probabilities: 50/800
Baseline probabilities: 52/800
Baseline probabilities: 54/800
Baseline probabilities: 56/800
Baseline probabilities: 58/800
Baseline probabilities: 60/800
Baseline probabilities: 62/800
Baseline probabilities: 64/800
Baseline pro

Baseline probabilities: 520/800
Baseline probabilities: 522/800
Baseline probabilities: 524/800
Baseline probabilities: 526/800
Baseline probabilities: 528/800
Baseline probabilities: 530/800
Baseline probabilities: 532/800
Baseline probabilities: 534/800
Baseline probabilities: 536/800
Baseline probabilities: 538/800
Baseline probabilities: 540/800
Baseline probabilities: 542/800
Baseline probabilities: 544/800
Baseline probabilities: 546/800
Baseline probabilities: 548/800
Baseline probabilities: 550/800
Baseline probabilities: 552/800
Baseline probabilities: 554/800
Baseline probabilities: 556/800
Baseline probabilities: 558/800
Baseline probabilities: 560/800
Baseline probabilities: 562/800
Baseline probabilities: 564/800
Baseline probabilities: 566/800
Baseline probabilities: 568/800
Baseline probabilities: 570/800
Baseline probabilities: 572/800
Baseline probabilities: 574/800
Baseline probabilities: 576/800
Baseline probabilities: 578/800
Baseline probabilities: 580/800
Baseline

,pair_id,label_word,feature_idx,label_specific_rank,stable_by_rule,baseline_label_specific_pre_gap,retention_rate_top_20,min_over_baseline,worst_variant_rank
0,0,World,13844,1,True,22.572865,1.000,0.876930,1
1,1,World,3994,2,True,17.208542,1.000,1.000000,2
2,2,World,14744,3,True,17.116394,1.000,0.938417,3
3,3,World,3163,4,True,10.695886,1.000,0.927058,6
4,4,World,5222,5,True,9.152500,1.000,0.989780,6
5,5,World,8901,6,True,9.130677,1.000,0.990756,6
6,6,World,406,7,True,8.718281,1.000,0.877048,8
7,7,World,1728,8,True,8.028021,1.000,0.911988,9
8,8,World,9069,9,True,7.454505,1.000,0.970708,10
9,9,World,5743,10,True,7.108073,1.000,0.775164,15


...

Stable versus LFH-sensitive causal summary:


,causal_group,n_label_feature_pairs,mean_active_prompt_fraction,median_active_prompt_fraction,mean_ablation_decrease,mean_ablation_decrease_ci_low,mean_ablation_decrease_ci_high,mean_ablation_decrease_pp,mean_ablation_decrease_ci_low_pp,mean_ablation_decrease_ci_high_pp,mean_amplification_increase,mean_amplification_increase_ci_low,mean_amplification_increase_ci_high,mean_amplification_increase_pp,mean_amplification_increase_ci_low_pp,mean_amplification_increase_ci_high_pp
0,multi-prefix stable,61,0.527459,0.535,0.000644,0.000346,0.001036,0.064441,0.034605,0.103626,0.007428,0.004637,0.010916,0.742773,0.463699,1.091553
1,LFH-sensitive,19,0.334474,0.335,0.000096,-0.000183,0.000466,0.009590,-0.018280,0.046616,0.001497,0.000167,0.002893,0.149667,0.016672,0.289307


Per-label stable/sensitive causal summary:


,label_word,causal_group,n_label_feature_pairs,mean_ablation_decrease_pp,mean_amplification_increase_pp,mean_active_prompt_fraction
0,Business,LFH-sensitive,3,0.015809,0.105750,0.295000
1,Business,multi-prefix stable,17,0.131560,1.477575,0.474706
2,Sports,LFH-sensitive,4,0.002445,-0.033353,0.046250
3,Sports,multi-prefix stable,16,0.016904,0.124588,0.590313
4,Technology,LFH-sensitive,7,0.019983,0.205865,0.355000
5,Technology,multi-prefix stable,13,0.055466,1.069752,0.446154
6,World,LFH-sensitive,5,-0.002975,0.243757,0.560000
7,World,multi-prefix stable,15,0.046857,0.286014,0.590667


Quick readout
Mean ablation decrease, stable group:      0.0644 percentage points
Mean ablation decrease, LFH-sensitive:     0.0096 percentage points
Mean amplification increase, stable group: 0.7428 percentage points
Mean amplification increase, sensitive:    0.1497 percentage points

Pattern matches the ideal validation: multi-prefix-stable features are more causally relevant on both tests.



## How to report the causal validation

The key table to include in the rebuttal is `causal_group_summary_df`:

- `mean_ablation_decrease_pp`: how much gold-label probability falls when the selected feature is removed when active.
- `mean_amplification_increase_pp`: how much gold-label probability rises when the selected feature direction is added with the current `CAUSAL_AMPLIFICATION_ALPHA`.

The intended coarse validation pattern is:

```text
multi-prefix stable features  >  LFH-sensitive features
```

for both ablation decrease and amplification increase. If the pattern holds, the rebuttal can state that the multi-prefix stability filter is not merely a descriptive diagnostic: it preferentially retains features with stronger causal support for the intended label, while filtering out single-prefix features that are more likely to be local-history-sensitive artifacts.



## How to report this experiment

A compact rebuttal/table can report:

- that features are selected by **label-specific preactivation gap**, not by overall activation magnitude;
- for each AGNews label, the top-20 single-prefix label-specific features;
- the fraction of those features that pass the multi-prefix stability rule;
- examples of high-ranked label-specific features that are unstable when only the final demonstration changes;
- Plotly heatmaps showing label-specific preactivation gaps across final-demo variants.

Suggested interpretation:

> We add an LFH-aware multi-prefix stability check for label-specific SAE features. For each fixed query, only the final demonstration is resampled. A feature selected from the original prefix is retained as robust only if its label-specific preactivation gap remains high across these alternative prefixes. Features whose apparent label-specificity collapses or moves sharply under final-demo variation are flagged as LFH-sensitive rather than directly interpreted as query-semantic features.

This is intended as a lightweight diagnostic, not a complete causal decomposition.
